# Notebook 20 - Empirical Event-Weight API

## Exercise 5: reusable estimator for canonical USDJPY target-event histories

# 1. Purpose and contract

Notebook 20 packages the frozen N16-N19 workflow as a reusable API. It inherits corrected Notebook 18 target-specific response days and uses target_model_day in all long-form target output. The N20 equivalence suite validates faithful reproduction of the frozen upstream methodology; it does not independently prove that the economic model or estimand is structurally correct.

## 2. Imports, paths, frozen configuration, and reference inputs

In [1]:
from pathlib import Path
import math, random
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mutual_info_score
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display
ROOT=next(x for x in [Path.cwd(),*Path.cwd().parents] if (x/"Data"/"processed").exists()); PROCESSED=ROOT/"Data"/"processed"
torch.set_num_threads(1); DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available(): torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
RF_GRID={"RF_01":{"max_depth":8,"min_samples_leaf":5},"RF_02":{"max_depth":8,"min_samples_leaf":10},"RF_03":{"max_depth":None,"min_samples_leaf":5},"RF_04":{"max_depth":None,"min_samples_leaf":10}}; GB_GRID={"GB_01":{"n_estimators":200,"learning_rate":.03},"GB_02":{"n_estimators":200,"learning_rate":.05},"GB_03":{"n_estimators":400,"learning_rate":.03},"GB_04":{"n_estimators":400,"learning_rate":.05}}; MLP_GRID={"MLP_01":([32],1e-3),"MLP_02":([32],3e-4),"MLP_03":([32,16],1e-3),"MLP_04":([32,16],3e-4)}; TR_GRID={"TR_01":((16,2,32),1e-3),"TR_02":((16,2,32),3e-4),"TR_03":((24,4,48),1e-3),"TR_04":((24,4,48),3e-4)}
CONFIG={"max_lag":12,"states":["Q","R","I"],"features":[f"{s}_lag{j}" for j in range(12,0,-1) for s in "QRI"],"gc_alpha":.05,"tdmi_bins":10,"tdmi_surrogates":1000,"neural_seeds":[19,119,219],"max_epochs":400,"patience":40,"min_delta":1e-5,"batch_size":64,"rf_grid":RF_GRID,"gb_grid":GB_GRID,"mlp_grid":MLP_GRID,"transformer_grid":TR_GRID,"bootstrap_reps":10000,"bootstrap_seed":19,"transformer_max_norm":1.0,"mlp_gradient_clipping":"none","equivalence_tolerances":{"parametric":1e-10,"tree":1e-10,"neural":1e-6}}
spec16=pd.read_csv(PROCESSED/"16_primary_specification.csv").iloc[0]; spec18=pd.read_csv(PROCESSED/"18_primary_specification.csv").iloc[0]; IV_COLUMN=str(spec16["iv_column"])
assert str(spec18["event_alignment_version"])=="TARGET_SPECIFIC_NY_CUT_1000_ET" and str(spec18["alignment_selection_sample"])=="TRAIN_ONLY" and str(spec18["bloomberg_snapshot_time_identified"]).lower() in {"false","0"}
raw_model_panel=pd.read_csv(PROCESSED/"16_empirical_analysis_panel.csv",parse_dates=["model_day"]); reference_mapping=pd.read_csv(PROCESSED/"18_target_specific_event_mapping.csv",parse_dates=["realised_model_day","implied_model_day","target_model_day","model_day","event_timestamp_utc"]); reference_n18=pd.read_csv(PROCESSED/"18_event_response_weights.csv",parse_dates=["realised_model_day","implied_model_day","target_model_day","model_day"]); reference_n19=pd.read_csv(PROCESSED/"19_event_response_weights.csv",parse_dates=["realised_model_day","implied_model_day","target_model_day","model_day"]); reference_n19_spec=pd.read_csv(PROCESSED/"19_selected_hyperparameters.csv"); reference_n19_ordinary=pd.read_csv(PROCESSED/"19_ordinary_response_ratios.csv",parse_dates=["model_day","target_model_day"])


## 3. Event input validation and audit helpers

Lags are constructed on intact chronology before an event day is masked. Multiple labels on one day remain separate output occurrences.

In [2]:
def prepare_model_panel(data,config=CONFIG):
    x=data.copy().sort_values("model_day").reset_index(drop=True); x["Q"]=pd.to_numeric(x["squared_return"]); x["R"]=pd.to_numeric(x["realised_variance_ann_252"]); x["I"]=pd.to_numeric(x[IV_COLUMN])
    for s in "QRI":
        for j in range(1,13): x[f"{s}_lag{j}"]=x[s].shift(j)
    for target in "RI": x[f"{target}_base36"]=np.isfinite(x[[target,*config["features"]]]).all(axis=1)
    assert x.model_day.is_unique and x.model_day.is_monotonic_increasing
    return x

def _events(events,panel,name):
    if not {"event_id","model_day","event_label"}.issubset(events): raise ValueError(f"{name} requires event_id, model_day, event_label")
    x=events.copy(); x["model_day"]=pd.to_datetime(x.model_day,errors="coerce")
    if x.event_id.isna().any() or x.event_id.duplicated().any() or x.model_day.isna().any(): raise ValueError(f"{name} has duplicate identifiers or invalid days")
    if not x.model_day.isin(panel.model_day).all(): raise ValueError(f"{name} contains days outside canonical panel support")
    return x.merge(panel[["model_day","sample_split"]],on="model_day",how="left",validate="many_to_one")

def _unique_eligible_dates(events,panel,features,eligibility_column):
    """One eligible predictor row per requested model day; occurrence multiplicity is never changed."""
    dates=events[["model_day"]].drop_duplicates().copy(); assert dates.model_day.is_unique
    columns=list(dict.fromkeys(["model_day",eligibility_column,*features])); x=dates.merge(panel[columns],on="model_day",how="left",validate="one_to_one")
    eligible=x[eligibility_column].fillna(False).astype(bool)&np.isfinite(x[features]).all(axis=1)
    out=x.loc[eligible,["model_day",*features]].copy(); assert out.model_day.is_unique
    return out

def _weight_rows(events,panel,forecast,target,model,stage,is_oos,eligibility_column):
    if "sample_split" not in events.columns: raise ValueError("Event occurrences must already contain canonical sample_split.")
    if not {"model_day","background_forecast"}.issubset(forecast.columns): raise ValueError("forecast must contain model_day and background_forecast")
    assert forecast.model_day.is_unique
    n_occurrences=len(events); x=events.merge(panel[["model_day",target,eligibility_column]],on="model_day",how="left",validate="many_to_one").rename(columns={target:"actual",eligibility_column:"equation_eligible"}).merge(forecast[["model_day","background_forecast"]],on="model_day",how="left",validate="many_to_one")
    assert len(x)==n_occurrences and "sample_split" in x.columns and "sample_split_x" not in x.columns and "sample_split_y" not in x.columns
    x["equation_eligible"]=x.equation_eligible.fillna(False).astype(bool); x["background_positive"]=x.equation_eligible & x.background_forecast.gt(0); x["weight_defined"]=x.equation_eligible & x.background_positive; x["event_weight"]=np.where(x.weight_defined,x.actual/x.background_forecast,np.nan); x["undefined_reason"]=np.select([~x.equation_eligible,x.equation_eligible & ~x.background_positive],["not_equation_eligible","nonpositive_background"],default="")
    x["target"],x["model"],x["forecast_stage"],x["is_out_of_sample"]=target,model,stage,is_oos
    return x[["event_id","model_day","event_label","sample_split","forecast_stage","is_out_of_sample","target","model","actual","background_forecast","equation_eligible","background_positive","weight_defined","event_weight","undefined_reason"]]


## 4. Step 1 - N16 dependence structure

In [3]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
CONFIG.update({"gc_max_lag":20,"gc_random_seed":16017,"tdmi_surrogates":1000,"tdmi_primary_bins":10})
N16_STATE_NAMES={"Q":"squared_return","R":"realised_variance_ann_252","I":"implied_variance_ann"}
N16_TDMI_EDGES=[("squared_return_to_realised_variance","Q","R"),("implied_variance_to_realised_variance","I","R"),("squared_return_to_implied_variance","Q","I"),("realised_variance_to_implied_variance","R","I")]
def _n16_lagcols(state,p): return [f"{state}_lag{j}" for j in range(1,p+1)]
def _n16_rows(panel,target,p,split):
    cols=[target]+sum((_n16_lagcols(state,p) for state in "QRI"),[]); return panel.loc[panel.sample_split.eq(split)&np.isfinite(panel[cols]).all(axis=1),cols].copy()
def _n16_holm(frame,alpha):
    out=frame.copy(); mask=out.raw_p_value.notna(); out["holm_adjusted_p_value"]=np.nan; out["holm_reject"]=False
    if mask.any():
        reject,padj,_,_=multipletests(out.loc[mask,"raw_p_value"],alpha=alpha,method="holm"); out.loc[mask,"holm_adjusted_p_value"]=padj; out.loc[mask,"holm_reject"]=reject
    return out
def _n16_gc(panel,target,source,p,split):
    rows=_n16_rows(panel,target,p,split); names=sum((_n16_lagcols(s,p) for s in "QRI"),[]); y=rows[target].to_numpy(float); xu=sm.add_constant(rows[names].to_numpy(float),has_constant="add"); source_names=_n16_lagcols(source,p); pos=[i for i,n in enumerate(["const",*names]) if n in source_names]; xr=xu[:,[i for i,n in enumerate(["const",*names]) if n not in source_names]]; restricted,unrestricted=sm.OLS(y,xr).fit(),sm.OLS(y,xu).fit(); q=len(pos); df=float(unrestricted.df_resid); f=max(0.,((restricted.ssr-unrestricted.ssr)/q)/(unrestricted.ssr/df)); hac=max(1,int(np.floor(4*(len(rows)/100)**(2/9)))); robust=unrestricted.get_robustcov_results(cov_type="HAC",maxlags=hac); restriction=np.zeros((q,len(names)+1)); restriction[np.arange(q),pos]=1; wald=robust.wald_test(restriction,scalar=True)
    return {"split":split,"target":target,"source":source,"history_order":p,"n_observations":len(rows),"restricted_RSS":float(restricted.ssr),"unrestricted_RSS":float(unrestricted.ssr),"classical_F":float(f),"raw_p_value":float(stats.f.sf(f,q,df)),"partial_R2":float((restricted.ssr-unrestricted.ssr)/restricted.ssr),"HAC_Wald_stat":float(np.asarray(wald.statistic).squeeze()),"HAC_p":float(np.asarray(wald.pvalue).squeeze()),"HAC_maxlags":hac}
def _n16_edges(values,bins):
    cuts=np.unique(np.quantile(np.asarray(values,float)[np.isfinite(values)],np.linspace(0,1,bins+1)[1:-1]))
    if len(cuts)<1: raise ValueError("TDMI requires at least two distinct training regions.")
    return np.r_[-np.inf,cuts,np.inf]
def _n16_mi(x,y,ex,ey):
    xb=np.digitize(x,ex[1:-1],right=True); yb=np.digitize(y,ey[1:-1],right=True); counts=np.zeros((xb.max()+1,yb.max()+1)); np.add.at(counts,(xb,yb),1); joint=counts/counts.sum(); product=joint.sum(1,keepdims=True)@joint.sum(0,keepdims=True); nz=joint>0; return float(np.sum(joint[nz]*np.log(joint[nz]/product[nz])))
def _n16_tdmi(panel,source,target,lag,split,edges,override=None):
    assert 1<=lag<=CONFIG["gc_max_lag"]
    base=panel[source].to_numpy(float) if override is None else override; lagged=np.roll(base,lag); current=panel[target].to_numpy(float); mask=panel.sample_split.eq(split).to_numpy(bool); mask[:lag]=False; valid=mask&np.isfinite(lagged)&np.isfinite(current); target_positions=np.flatnonzero(valid); assert np.all(target_positions-lag<target_positions) and panel.loc[valid,"sample_split"].eq(split).all()
    return (_n16_mi(lagged[valid],current[valid],edges[source],edges[target]),int(valid.sum())) if valid.sum()>=20 else (np.nan,int(valid.sum()))
def _n16_surrogate(panel,source,target,lag,split,edges,rng,config):
    observed,n=_n16_tdmi(panel,source,target,lag,split,edges); positions=np.flatnonzero(panel.sample_split.eq(split)); segment=panel[source].to_numpy(float)[positions].copy(); shifts=np.arange(config["gc_max_lag"]+1,max(config["gc_max_lag"]+2,len(segment)-config["gc_max_lag"])); null=np.empty(config["tdmi_surrogates"])
    for i,shift in enumerate(rng.choice(shifts,size=config["tdmi_surrogates"])):
        altered=panel[source].to_numpy(float).copy(); altered[positions]=np.roll(segment,int(shift)); null[i]=_n16_tdmi(panel,source,target,lag,split,edges,altered)[0]
    return observed,n,float((1+np.sum(null>=observed))/(1+len(null)))
def _n16_tdmi_training_search(panel,edge_id,source,target,bins,with_surrogates,config):
    edges={source:_n16_edges(panel.loc[panel.sample_split.eq("train")&np.isfinite(panel[source]),source],bins),target:_n16_edges(panel.loc[panel.sample_split.eq("train")&np.isfinite(panel[target]),target],bins)}; rows=[]; rng=np.random.default_rng(config["gc_random_seed"]+bins+len(N16_STATE_NAMES[source])+len(N16_STATE_NAMES[target]))
    for lag in range(1,config["gc_max_lag"]+1):
        value,n=_n16_tdmi(panel,source,target,lag,"train",edges); row={"edge_id":edge_id,"source":source,"target":target,"split":"train","lag":lag,"bins":bins,"tdmi":value,"n_pairs":n,"raw_p_value":np.nan}
        if with_surrogates: value,n,pv=_n16_surrogate(panel,source,target,lag,"train",edges,rng,config); row.update(tdmi=value,n_pairs=n,raw_p_value=pv)
        rows.append(row)
    out=pd.DataFrame(rows)
    if with_surrogates: out=_n16_holm(out,config["gc_alpha"]); out["significant"]=out.holm_reject
    else: out["significant"]=np.nan
    return out,edges


In [4]:
def estimate_dependence_structure(model_panel,config=CONFIG):
    """Faithful N16 order scan, selected-order conditional GC, Holm, and diagnostic TDMI."""
    panel=prepare_model_panel(model_panel,config)
    for state in "QRI":
        for lag in range(13,config["gc_max_lag"]+1): panel[f"{state}_lag{lag}"]=panel[state].shift(lag)
    scans=[]
    for target in "RI":
        common=_n16_rows(panel,target,config["gc_max_lag"],"train")
        for p in range(1,config["gc_max_lag"]+1):
            names=sum((_n16_lagcols(s,p) for s in "QRI"),[]); fit=sm.OLS(common[target],sm.add_constant(common[names],has_constant="add")).fit(); scans.append({"target":target,"history_order":p,"AIC":fit.aic,"BIC":fit.bic,"n_observations":len(common)})
    scan=pd.DataFrame(scans); selected={"R":int(scan.loc[scan.target.eq("R")].sort_values(["BIC","history_order"]).iloc[0].history_order),"I":int(scan.loc[scan.target.eq("I")].sort_values(["BIC","history_order"]).iloc[0].history_order)}
    gc=pd.DataFrame([_n16_gc(panel,target,source,selected[target],split) for target,source in [("R","Q"),("R","I"),("I","Q"),("I","R")] for split in ["train","validation"]]); gc["holm_adjusted_p_value"]=np.nan; gc["holm_reject"]=False
    for split in ["train","validation"]:
        ix=gc.split.eq(split); adjusted=_n16_holm(gc.loc[ix],config["gc_alpha"]); gc.loc[ix,["holm_adjusted_p_value","holm_reject"]]=adjusted[["holm_adjusted_p_value","holm_reject"]]
    retained=gc.groupby(["source","target"],as_index=False).agg(retained=("holm_reject","all"),train_raw_p=("raw_p_value","first"),validation_raw_p=("raw_p_value","last"),train_holm_p=("holm_adjusted_p_value","first"),validation_holm_p=("holm_adjusted_p_value","last"))
    primary=[]; summaries=[]; holdouts=[]; sensitivity=[]
    for edge_id,source,target in N16_TDMI_EDGES:
        curve,edges=_n16_tdmi_training_search(panel,edge_id,source,target,config["tdmi_primary_bins"],True,config); primary.append(curve.assign(curve_role="primary_10_bin")); sig=curve.loc[curve.significant.fillna(False)]; peak_row=sig.sort_values("tdmi",ascending=False).iloc[0] if len(sig) else None; peak=int(peak_row.lag) if peak_row is not None else np.nan
        summaries.append({"edge_id":edge_id,"source":source,"target":target,"training_tdmi_support":bool(len(sig)),"training_tdmi_peak_lag":peak,"training_tdmi_value":float(peak_row.tdmi) if peak_row is not None else np.nan,"training_tdmi_raw_p":float(peak_row.raw_p_value) if peak_row is not None else np.nan,"training_tdmi_holm_p":float(peak_row.holm_adjusted_p_value) if peak_row is not None else np.nan})
        if np.isfinite(peak):
            for split in ["validation","test"]:
                value,n,pv=_n16_surrogate(panel,source,target,int(peak),split,edges,np.random.default_rng(config["gc_random_seed"]+500+len(edge_id)),config); holdouts.append({"edge_id":edge_id,"split":split,"tdmi_value":value,"tdmi_p":pv,"tdmi_support":pv<config["gc_alpha"],"n_pairs":n})
        for bins in [5,10,15]:
            candidate,_=_n16_tdmi_training_search(panel,edge_id,source,target,bins,bins==config["tdmi_primary_bins"],config); eligible=candidate.loc[candidate.significant.fillna(False)] if bins==config["tdmi_primary_bins"] else candidate; sensitivity.append({"edge_id":edge_id,"bins":bins,"peak_lag":int(eligible.sort_values("tdmi",ascending=False).iloc[0].lag) if len(eligible) else np.nan,"peak_is_primary_significant":bool(bins==config["tdmi_primary_bins"] and len(eligible))})
    tdmi=pd.concat(primary,ignore_index=True); tdmi_summary=pd.DataFrame(summaries); tdmi_holdout=pd.DataFrame(holdouts); tdmi_sensitivity=pd.DataFrame(sensitivity)
    for split in ["validation","test"]:
        lookup=tdmi_holdout.loc[tdmi_holdout.split.eq(split)].set_index("edge_id") if len(tdmi_holdout) else pd.DataFrame(); tdmi_summary[f"tdmi_{split}_value"]=tdmi_summary.edge_id.map(lookup.tdmi_value if len(lookup) else {}); tdmi_summary[f"tdmi_{split}_p"]=tdmi_summary.edge_id.map(lookup.tdmi_p if len(lookup) else {}); tdmi_summary[f"tdmi_{split}_support"]=tdmi_summary.edge_id.map(lookup.tdmi_support if len(lookup) else {}).fillna(False).astype(bool)
    return {"gc_results":gc,"tdmi_results":tdmi,"tdmi_summary":tdmi_summary,"tdmi_holdout":tdmi_holdout,"tdmi_bin_sensitivity":tdmi_sensitivity,"order_scan":scan,"selected_orders":selected,"retained_structure":retained,"metadata":{"procedure":"N16 selected-order GC with Holm and direct positive-lag fixed-bin circular-shift TDMI","event_history_used":False,"tdmi_primary_bins":config["tdmi_primary_bins"],"tdmi_surrogates":config["tdmi_surrogates"]}}


## 5. Step 2 - N17 ordinary parametric calibration

In [5]:
def _parametric_features(structure,target):
    """Use N16's selected target order and retained directed edges, not a hard-coded N17 formula."""
    if not {"selected_orders","retained_structure"}.issubset(structure): raise ValueError("structure must contain selected_orders and retained_structure")
    p=int(structure["selected_orders"][target]); edges=structure["retained_structure"].set_index(["source","target"])["retained"].to_dict(); features=[]
    for source in "QRI":
        if source==target or bool(edges.get((source,target),False)): features.extend(_n16_lagcols(source,p))
    return features

def fit_parametric_models(model_panel,structure,config=CONFIG):
    panel=prepare_model_panel(model_panel,config); pieces=[]; coeffs=[]; models={}
    for target in "RI":
        features=_parametric_features(structure,target); eligibility=f"{target}_parametric_eligible"; panel[eligibility]=np.isfinite(panel[[target,"Q_lag1","R_lag1","I_lag1"]]).all(axis=1) if target=="R" else np.isfinite(panel[[target,*config["features"]]]).all(axis=1)
        train=panel.loc[panel.sample_split.eq("train")&panel[eligibility]&np.isfinite(panel[features]).all(axis=1)]; train_model=sm.OLS(train[target],sm.add_constant(train[features],has_constant="add")).fit(); coeffs.append(pd.DataFrame({"target":target,"predictor":train_model.params.index,"coefficient":train_model.params.values,"calibration_stage":"TRAIN"}))
        valid=panel.loc[panel.sample_split.eq("validation")&panel[eligibility]&np.isfinite(panel[features]).all(axis=1)]; pieces.append(pd.DataFrame({"model_day":valid.model_day,"target":target,"split":"validation","forecast_stage":"TRAIN_TO_VALIDATION","actual":valid[target],"background_forecast":train_model.predict(sm.add_constant(valid[features],has_constant="add"))}))
        final=panel.loc[panel.sample_split.isin(["train","validation"])&panel[eligibility]&np.isfinite(panel[features]).all(axis=1)]; final_model=sm.OLS(final[target],sm.add_constant(final[features],has_constant="add")).fit(); models[target]={"train":train_model,"train_validation":final_model,"features":features}; coeffs.append(pd.DataFrame({"target":target,"predictor":final_model.params.index,"coefficient":final_model.params.values,"calibration_stage":"TRAIN_VALIDATION"}))
        test=panel.loc[panel.sample_split.eq("test")&panel[eligibility]&np.isfinite(panel[features]).all(axis=1)]; pieces.append(pd.DataFrame({"model_day":test.model_day,"target":target,"split":"test","forecast_stage":"TRAIN_VALIDATION_TO_TEST","actual":test[target],"background_forecast":final_model.predict(sm.add_constant(test[features],has_constant="add"))}))
    forecasts=pd.concat(pieces,ignore_index=True); metrics=forecasts.groupby(["target","split","forecast_stage"]).apply(lambda x:pd.Series(_metrics(x.actual,x.background_forecast)),include_groups=False).reset_index(); return {"coefficients":pd.concat(coeffs,ignore_index=True),"forecasts":forecasts,"metrics":metrics,"models":models,"metadata":{"event_history_used":False,"active_regressors_from_structure":True}}


## 6. Common eventless calibration and Step 3 parametric weights

In [6]:
def build_eventless_calibration_sample(model_panel,event_history,evaluation_events=None,config=CONFIG):
    panel=prepare_model_panel(model_panel,config) if "R_lag1" not in model_panel else model_panel.copy(); history=_events(event_history,panel,"event_history"); evaluation=history.copy() if evaluation_events is None else _events(evaluation_events,panel,"evaluation_events")
    identity=["event_id","model_day","event_label"]; reference=history.set_index("event_id")[identity[1:]]
    if not set(evaluation.event_id).issubset(set(history.event_id)) or not evaluation.set_index("event_id")[identity[1:]].equals(reference.loc[evaluation.event_id].set_axis(evaluation.event_id,axis=0)): raise ValueError("evaluation_events must match event_history on event_id, model_day, and event_label")
    # N17's R support is intentionally broader than its selected active regressors.
    panel["R_parametric_eligible"]=np.isfinite(panel[["R","Q_lag1","R_lag1","I_lag1"]]).all(axis=1); panel["I_parametric_eligible"]=np.isfinite(panel[["I",*config["features"]]]).all(axis=1); panel["is_target_event_day"]=panel.model_day.isin(set(history.model_day))
    for target in "RI": panel[f"{target}_eventless_parametric"]=panel[f"{target}_parametric_eligible"] & ~panel.is_target_event_day; panel[f"{target}_eventless36"]=panel[f"{target}_base36"] & ~panel.is_target_event_day
    audit={"n_target_events_supplied":len(history),"n_target_event_days":history.model_day.nunique(),"n_target_event_days_masked_from_calibration":history.model_day.nunique(),"n_target_events_evaluated":len(evaluation)}
    for split in ["train","validation","test"]: audit[f"n_target_events_in_{split}"]=int(history.sample_split.eq(split).sum()); audit[f"n_evaluation_events_{split}"]=int(evaluation.sample_split.eq(split).sum())
    return {"panel":panel,"event_history":history,"evaluation_events":evaluation,"audit":pd.DataFrame([audit])}

def _eventless_ols(panel,target,features,splits):
    eligible=panel[f"{target}_eventless_parametric"]&panel.sample_split.isin(splits)&np.isfinite(panel[features]).all(axis=1); x=panel.loc[eligible]; return sm.OLS(x[target],sm.add_constant(x[features],has_constant="add")).fit()
def estimate_parametric_event_weights(model_panel,event_history,evaluation_events=None,structure=None,config=CONFIG):
    structure=estimate_dependence_structure(model_panel,config) if structure is None else structure; built=build_eventless_calibration_sample(model_panel,event_history,evaluation_events,config); panel=built["panel"]; output=[]; coefficients=[]; fitted={}
    for target in "RI":
        features=_parametric_features(structure,target); train=_eventless_ols(panel,target,features,["train"]); final=_eventless_ols(panel,target,features,["train","validation"]); fitted[target]={"train":train,"train_validation":final,"features":features}; coefficients += [pd.DataFrame({"target":target,"predictor":train.params.index,"coefficient":train.params.values,"calibration_stage":"EVENTLESS_TRAIN"}),pd.DataFrame({"target":target,"predictor":final.params.index,"coefficient":final.params.values,"calibration_stage":"EVENTLESS_TRAIN_VALIDATION"})]
        for split,model,stage,oos in [("train",train,"EVENTLESS_TRAIN_IN_SAMPLE_DIAGNOSTIC",False),("validation",train,"EVENTLESS_TRAIN_TO_VALIDATION",True),("test",final,"EVENTLESS_TRAIN_VALIDATION_TO_TEST",True)]:
            events=built["evaluation_events"].loc[built["evaluation_events"].sample_split.eq(split)]; date_rows=_unique_eligible_dates(events,panel,features,f"{target}_parametric_eligible"); forecast=pd.DataFrame({"model_day":date_rows.model_day,"background_forecast":model.predict(sm.add_constant(date_rows[features],has_constant="add")) if len(date_rows) else np.array([],dtype=float)}); assert forecast.model_day.is_unique; output.append(_weight_rows(events,panel,forecast,target,"PARAMETRIC_"+target,stage,oos,f"{target}_parametric_eligible"))
    return {"event_weights":pd.concat(output,ignore_index=True),"coefficients":pd.concat(coefficients,ignore_index=True),"models":fitted,"event_audit":built["audit"],"metadata":{"eventless_parameters_refit":True,"active_regressors_from_structure":True}}


## 7. Flexible model definitions and Step 4 selection

In [7]:
class MLPRegressor(nn.Module):
    def __init__(self,hidden):
        super().__init__(); layers=[]; width=36
        for units in hidden: layers += [nn.Linear(width,units),nn.ReLU(),nn.Dropout(.1)]; width=units
        layers.append(nn.Linear(width,1)); self.network=nn.Sequential(*layers)
    def forward(self,x): return self.network(x).squeeze(-1)
class TransformerRegressor(nn.Module):
    def __init__(self,d,h,ff):
        super().__init__(); self.embed=nn.Linear(3,d); pos=torch.zeros(12,d); position=torch.arange(12).unsqueeze(1); divisor=torch.exp(torch.arange(0,d,2)*(-math.log(10000)/d)); pos[:,0::2]=torch.sin(position*divisor); pos[:,1::2]=torch.cos(position*divisor); self.register_buffer("pos",pos.unsqueeze(0)); self.register_buffer("mask",torch.triu(torch.ones(12,12,dtype=torch.bool),1)); self.encoder=nn.TransformerEncoder(nn.TransformerEncoderLayer(d,h,ff,dropout=.1,activation="gelu",batch_first=True),1); self.head=nn.Linear(d,1)
    def forward(self,x): return self.head(self.encoder(self.embed(x)+self.pos,mask=self.mask)[:,-1]).squeeze(-1)
def _seed(seed): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed) if torch.cuda.is_available() else None
def fit_state_scaler(frame,target,stage):
    scale={}; rows=[]
    for s in "QRI":
        z=frame[[f"{s}_lag{j}" for j in range(1,13)]].to_numpy(float).ravel(); scale[s]=(z.mean(),z.std()); rows.append({"target":target,"stage":stage,"state":s,"mean":z.mean(),"std":z.std()})
    z=frame[target].to_numpy(float); scale["TARGET"]=(z.mean(),z.std()); rows.append({"target":target,"stage":stage,"state":"TARGET","mean":z.mean(),"std":z.std()}); return scale,rows
def _x(frame,scale,family,config):
    # N19 arithmetic is float64 through standardisation; only tensor construction casts to float32.
    z=frame[config["features"]].to_numpy(dtype=float).copy()
    for i,f in enumerate(config["features"]):
        mean,std=scale[f[0]]
        if not np.isfinite(mean) or not np.isfinite(std) or std<=0: raise ValueError(f"Invalid neural feature scale for {f[0]}")
        z[:,i]=(z[:,i]-mean)/std
    assert np.isfinite(z).all(); features=torch.tensor(z,dtype=torch.float32)
    return features.reshape(-1,12,3) if family=="TRANSFORMER" else features
def _target_tensor(frame,target,target_scale):
    # Avoid the pandas→PyTorch incompatibility while preserving N19 float64 scaling arithmetic.
    mean,std=target_scale
    if not np.isfinite(mean) or not np.isfinite(std) or std<=0: raise ValueError("Invalid neural target scale")
    values=frame[target].to_numpy(dtype=float); values=np.asarray((values-mean)/std,dtype=float); assert values.ndim==1 and np.isfinite(values).all()
    return torch.tensor(values,dtype=torch.float32)
def _model(family,configuration): return MLPRegressor(configuration[0]) if family=="MLP" else TransformerRegressor(*configuration[0])
def train_neural_model(family,configuration,seed,train,validation,target,scale,config,epochs=None):
    _seed(seed); model=_model(family,configuration).to(DEVICE); optimizer=torch.optim.AdamW(model.parameters(),lr=configuration[1],weight_decay=1e-4); loss=nn.MSELoss(); train_x=_x(train,scale,family,config); train_y=_target_tensor(train,target,scale["TARGET"]); assert train_x.dtype==torch.float32 and train_y.dtype==torch.float32; vx=_x(validation,scale,family,config).to(DEVICE); vy=_target_tensor(validation,target,scale["TARGET"]).to(DEVICE); assert vx.dtype==torch.float32 and vy.dtype==torch.float32; loader=DataLoader(TensorDataset(train_x,train_y),batch_size=config["batch_size"],shuffle=True); best=np.inf; state=None; best_epoch=0; wait=0
    for epoch in range(1,(epochs or config["max_epochs"])+1):
        model.train()
        for bx,by in loader:
            optimizer.zero_grad(); value=loss(model(bx.to(DEVICE)),by.to(DEVICE)); value.backward()
            if family=="TRANSFORMER": torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
            optimizer.step()
        if epochs is not None: continue
        model.eval()
        with torch.no_grad(): value=loss(model(vx),vy).item()
        if value < best-config["min_delta"]: best=value; state={k:v.detach().clone() for k,v in model.state_dict().items()}; best_epoch=epoch; wait=0
        else: wait += 1
        if wait >= config["patience"]: break
    if epochs is None: model.load_state_dict(state)
    return model,(epochs if epochs is not None else best_epoch)
def _predict_neural(models,frame,scale,family,config):
    x=_x(frame,scale,family,config).to(DEVICE); out=[]
    for model in models:
        model.eval()
        with torch.no_grad(): out.append(model(x).detach().cpu().numpy()*scale["TARGET"][1]+scale["TARGET"][0])
    return np.mean(out,axis=0)

def select_flexible_models(model_panel,event_history,config=CONFIG):
    built=build_eventless_calibration_sample(model_panel,event_history,config=config); panel=built["panel"]; selected={}; metrics=[]; seed_rows=[]; features=config["features"]
    for target in "RI":
        train=panel.loc[panel.sample_split.eq("train")&panel[f"{target}_eventless36"]]; validation=panel.loc[panel.sample_split.eq("validation")&panel[f"{target}_base36"]&~panel.is_target_event_day]
        for name,grid,klass,fixed in [("RANDOM_FOREST",config["rf_grid"],RandomForestRegressor,{"n_estimators":500,"max_features":.5,"criterion":"squared_error","bootstrap":True,"random_state":19,"n_jobs":-1}),("GRADIENT_BOOSTING",config["gb_grid"],GradientBoostingRegressor,{"max_depth":2,"min_samples_leaf":10,"subsample":1.,"loss":"squared_error","random_state":19})]:
            candidates=[]
            for cid,params in grid.items():
                fitted=klass(**fixed,**params).fit(train[features],train[target]); candidates.append((_metrics(validation[target],fitted.predict(validation[features]))["RMSE"],cid))
            rmse,cid=min(candidates); selected[(target,name)]={"config_id":cid,"configuration":grid[cid],"final_epochs":None}; metrics.append({"target":target,"model_family":name,"selected_config":cid,"validation_RMSE":rmse})
        scale,_=fit_state_scaler(train,target,"TRAIN_SELECTION")
        for name,grid in [("MLP",config["mlp_grid"]),("TRANSFORMER",config["transformer_grid"])]:
            candidates=[]
            for cid,conf in grid.items():
                rmses=[]; bests=[]
                for seed in config["neural_seeds"]:
                    fitted,best=train_neural_model(name,conf,seed,train,validation,target,scale,config); rmse=_metrics(validation[target],_predict_neural([fitted],validation,scale,name,config))["RMSE"]; rmses.append(rmse); bests.append(best); seed_rows.append({"target":target,"model_family":name,"config_id":cid,"seed":seed,"best_epoch":best,"validation_RMSE_original_units":rmse})
                simplicity=len(conf[0]) if name=="MLP" else conf[0][0]; candidates.append((np.mean(rmses),simplicity,cid,int(np.median(bests))))
            rmse,_simplicity,cid,epochs=min(candidates); selected[(target,name)]={"config_id":cid,"configuration":grid[cid],"final_epochs":epochs}; metrics.append({"target":target,"model_family":name,"selected_config":cid,"validation_RMSE":rmse,"final_fixed_epochs":epochs})
    return {"selected":selected,"selected_hyperparameters":pd.DataFrame(metrics),"neural_seed_diagnostics":pd.DataFrame(seed_rows),"metadata":{"selection":"eventless TRAIN; ordinary VALIDATION; no TEST"}}

def _metrics(y,p):
    error=np.asarray(y)-np.asarray(p)
    return {"N":len(y),"RMSE":float(np.sqrt(np.mean(error**2))),"MAE":float(np.mean(np.abs(error)))}


## 8. Step 4 eventless refitting, pooling, and public API

In [8]:
def _flexible_predictions(model,frame,features,family,scale,config):
    assert frame.model_day.is_unique
    if frame.empty: return pd.DataFrame(columns=["model_day","background_forecast"])
    values=_predict_neural(model,frame,scale,family,config) if family in {"MLP","TRANSFORMER"} else model.predict(frame[features]); out=pd.DataFrame({"model_day":frame.model_day,"background_forecast":values}); assert out.model_day.is_unique; return out
def estimate_flexible_event_weights(model_panel,event_history,evaluation_events=None,flexible_spec=None,config=CONFIG):
    """Refit every supplied selected specification by requested forecast stage; occurrence rows are retained."""
    flexible_spec=select_flexible_models(model_panel,event_history,config) if flexible_spec is None else flexible_spec; built=build_eventless_calibration_sample(model_panel,event_history,evaluation_events,config); panel=built["panel"]; features=config["features"]; rows=[]; fitted={}; scaler_rows=[]; forecast_rows=[]
    stages=[("train",["train"],"EVENTLESS_TRAIN_IN_SAMPLE_DIAGNOSTIC",False),("validation",["train"],"EVENTLESS_TRAIN_TO_VALIDATION",True),("test",["train","validation"],"EVENTLESS_TRAIN_VALIDATION_TO_TEST",True)]
    tree_specs=[("RANDOM_FOREST",RandomForestRegressor,{"n_estimators":500,"max_features":.5,"criterion":"squared_error","bootstrap":True,"random_state":19,"n_jobs":-1}),("GRADIENT_BOOSTING",GradientBoostingRegressor,{"max_depth":2,"min_samples_leaf":10,"subsample":1.,"loss":"squared_error","random_state":19})]
    for target in "RI":
        for split,calibration_splits,stage,is_oos in stages:
            event_rows=built["evaluation_events"].loc[built["evaluation_events"].sample_split.eq(split)]
            if event_rows.empty: continue
            calibration=panel.loc[panel.sample_split.isin(calibration_splits)&panel[f"{target}_eventless36"]]; eligible_events=_unique_eligible_dates(event_rows,panel,features,f"{target}_base36"); forecast_panel=panel.loc[panel.sample_split.eq(split)&panel[f"{target}_base36"]&np.isfinite(panel[features]).all(axis=1)]; assert forecast_panel.model_day.is_unique
            for name,klass,fixed in tree_specs:
                spec=flexible_spec["selected"][(target,name)]; model=klass(**fixed,**spec["configuration"]).fit(calibration[features],calibration[target]); fitted[(target,name,stage)]=model; event_forecast=_flexible_predictions(model,eligible_events,features,name,None,config); assert event_forecast.model_day.is_unique; rows.append(_weight_rows(event_rows,panel,event_forecast,target,name,stage,is_oos,f"{target}_base36")); full_forecast=_flexible_predictions(model,forecast_panel,features,name,None,config); full_forecast["target"],full_forecast["model"],full_forecast["split"],full_forecast["forecast_stage"]=target,name,split,stage; forecast_rows.append(full_forecast)
            scale,audit=fit_state_scaler(calibration,target,stage); scaler_rows += audit
            for name in ["MLP","TRANSFORMER"]:
                spec=flexible_spec["selected"][(target,name)]; models=[train_neural_model(name,spec["configuration"],seed,calibration,calibration,target,scale,config,epochs=spec["final_epochs"])[0] for seed in config["neural_seeds"]]; fitted[(target,name,stage)]=models; event_forecast=_flexible_predictions(models,eligible_events,features,name,scale,config); assert event_forecast.model_day.is_unique; rows.append(_weight_rows(event_rows,panel,event_forecast,target,name,stage,is_oos,f"{target}_base36")); full_forecast=_flexible_predictions(models,forecast_panel,features,name,scale,config); full_forecast["target"],full_forecast["model"],full_forecast["split"],full_forecast["forecast_stage"]=target,name,split,stage; forecast_rows.append(full_forecast)
    return {"event_weights":pd.concat(rows,ignore_index=True),"forecasts":pd.concat(forecast_rows,ignore_index=True),"models":fitted,"neural_scalers":pd.DataFrame(scaler_rows),"event_audit":built["audit"],"metadata":{"eventless_parameters_refit":True,"scalers_refit":True,"stage_specific_refits":True,"empty_stages_skipped":True}}

def pool_event_weights(event_weights,config=CONFIG):
    """Pool separately by sample split and forecast stage; never mix TRAIN diagnostics with OOS weights."""
    rng=np.random.default_rng(config["bootstrap_seed"]); output=[]; group_keys=["event_label","sample_split","forecast_stage","is_out_of_sample","target","model"]
    for (label,split,stage,is_oos,target,model),x in event_weights.groupby(group_keys):
        valid=x.loc[x.weight_defined&np.isfinite(x.event_weight),"event_weight"].to_numpy(float); ci=(np.nan,np.nan) if not len(valid) else tuple(np.quantile(np.median(rng.choice(valid,size=(config["bootstrap_reps"],len(valid)),replace=True),axis=1),[.025,.975])); output.append({"event_label":label,"sample_split":split,"forecast_stage":stage,"is_out_of_sample":is_oos,"target":target,"model":model,"n_occurrences":len(x),"n_eligible":int(x.equation_eligible.sum()),"n_valid":len(valid),"valid_fraction":len(valid)/len(x),"q25":np.quantile(valid,.25) if len(valid) else np.nan,"median":np.median(valid) if len(valid) else np.nan,"q75":np.quantile(valid,.75) if len(valid) else np.nan,"mean":np.mean(valid) if len(valid) else np.nan,"median_ci_low":ci[0],"median_ci_high":ci[1],"bootstrap_reps":config["bootstrap_reps"],"bootstrap_seed":config["bootstrap_seed"]})
    return pd.DataFrame(output)

def estimate_event_weights(model_panel,event_history,evaluation_events=None,structure=None,flexible_spec=None,config=None):
    """Public Exercise-5 API for canonical USDJPY target-event histories."""
    cfg=CONFIG if config is None else config; structure_source="SUPPLIED" if structure is not None else "ESTIMATED_THIS_RUN"; flexible_source="SUPPLIED" if flexible_spec is not None else "SELECTED_THIS_RUN"; structure=estimate_dependence_structure(model_panel,cfg) if structure is None else structure; ordinary=fit_parametric_models(model_panel,structure,cfg); parametric=estimate_parametric_event_weights(model_panel,event_history,evaluation_events,structure,cfg); flexible_spec=select_flexible_models(model_panel,event_history,cfg) if flexible_spec is None else flexible_spec; flexible=estimate_flexible_event_weights(model_panel,event_history,evaluation_events,flexible_spec,cfg); built=build_eventless_calibration_sample(model_panel,event_history,evaluation_events,cfg); weights=pd.concat([parametric["event_weights"],flexible["event_weights"]],ignore_index=True); meta=pd.DataFrame([{**built["audit"].iloc[0].to_dict(),"structure_source":structure_source,"flexible_spec_source":flexible_source,"api_mode":"FULL" if structure_source=="ESTIMATED_THIS_RUN" and flexible_source=="SELECTED_THIS_RUN" else "REUSE_OR_PARTIAL_REUSE","models_estimated":"PARAMETRIC,RANDOM_FOREST,GRADIENT_BOOSTING,MLP,TRANSFORMER","neural_seeds":"19,119,219"}])
    return {"structure":structure,"parametric":ordinary,"parametric_eventless":parametric,"flexible_spec":flexible_spec,"flexible_eventless":flexible,"event_weights":weights,"pooled_weights":pool_event_weights(weights,cfg),"diagnostics":{"event_history_audit":built["audit"]},"run_metadata":meta}


In [9]:
_pool_test_config={**CONFIG,"bootstrap_reps":20}
_pool_test_weights=pd.DataFrame([{"event_label":"stage-pooling smoke test","sample_split":"train","forecast_stage":"EVENTLESS_TRAIN_IN_SAMPLE_DIAGNOSTIC","is_out_of_sample":False,"target":"R","model":"PARAMETRIC_R","equation_eligible":True,"weight_defined":True,"event_weight":1.0},{"event_label":"stage-pooling smoke test","sample_split":"train","forecast_stage":"EVENTLESS_TRAIN_IN_SAMPLE_DIAGNOSTIC","is_out_of_sample":False,"target":"R","model":"PARAMETRIC_R","equation_eligible":True,"weight_defined":True,"event_weight":3.0},{"event_label":"stage-pooling smoke test","sample_split":"test","forecast_stage":"EVENTLESS_TRAIN_VALIDATION_TO_TEST","is_out_of_sample":True,"target":"R","model":"PARAMETRIC_R","equation_eligible":True,"weight_defined":True,"event_weight":10.0},{"event_label":"stage-pooling smoke test","sample_split":"test","forecast_stage":"EVENTLESS_TRAIN_VALIDATION_TO_TEST","is_out_of_sample":True,"target":"R","model":"PARAMETRIC_R","equation_eligible":True,"weight_defined":True,"event_weight":14.0}])
_pool_test_result=pool_event_weights(_pool_test_weights,_pool_test_config)
assert len(_pool_test_result)==2
_pool_test_train=_pool_test_result.loc[_pool_test_result.sample_split.eq("train")].iloc[0]; _pool_test_test=_pool_test_result.loc[_pool_test_result.sample_split.eq("test")].iloc[0]
assert _pool_test_train.n_occurrences==2 and _pool_test_test.n_occurrences==2
assert _pool_test_train.n_valid==2 and _pool_test_test.n_valid==2
assert _pool_test_train["median"]==2.0 and _pool_test_test["median"]==12.0
assert not _pool_test_train.is_out_of_sample and _pool_test_test.is_out_of_sample


In [10]:
# Target-specific event adapter; authoritative N20 runtime plumbing.

DAY = {
    "R": "realised_model_day",
    "I": "implied_model_day",
}

PMODEL = {
    "R": "N18_SPARSE_LINEAR",
    "I": "N18_LINEAR36",
}

FLEX = [
    "RANDOM_FOREST",
    "GRADIENT_BOOSTING",
    "MLP",
    "TRANSFORMER",
]

ALIGNMENT_VERSION = "TARGET_SPECIFIC_NY_CUT_1000_ET"


def _norm(x, panel, name):
    need = {
        "event_id",
        "event_label",
        "realised_model_day",
        "implied_model_day",
    }

    if not need.issubset(x):
        raise ValueError(f"{name} schema")

    x = x.copy()

    for c in [
        "realised_model_day",
        "implied_model_day",
    ]:
        x[c] = pd.to_datetime(
            x[c],
            errors="coerce",
        )

    if "event_timestamp" not in x:
        x["event_timestamp"] = pd.NaT

    x["event_timestamp"] = pd.to_datetime(
        x.event_timestamp,
        utc=True,
        errors="coerce",
    )

    if (
        x.event_id.isna().any()
        or x.event_id.duplicated().any()
        or x[
            [
                "realised_model_day",
                "implied_model_day",
            ]
        ].isna().any().any()
    ):
        raise ValueError(f"{name} identity")

    if (
        not x.realised_model_day.isin(
            panel.model_day
        ).all()
        or not x.implied_model_day.isin(
            panel.model_day
        ).all()
    ):
        raise ValueError(f"{name} support")

    return x


def _long(x, p, t):
    z = x.copy()

    z["target"] = t
    z["target_model_day"] = z[DAY[t]]
    z["model_day"] = z.target_model_day

    z["sample_split"] = (
        z.target_model_day.map(
            p.set_index(
                "model_day"
            ).sample_split
        )
    )

    g = (
        z.groupby(
            "target_model_day",
            as_index=False,
        )
        .agg(
            n_target_occurrences=(
                "event_id",
                "size",
            ),
            n_target_families=(
                "event_label",
                "nunique",
            ),
            family_set=(
                "event_label",
                lambda q: " | ".join(
                    sorted(
                        set(
                            map(str, q)
                        )
                    )
                ),
            ),
        )
    )

    z = z.merge(
        g,
        on="target_model_day",
        validate="many_to_one",
    )

    z["is_clean_single_family"] = (
        z.n_target_families.eq(1)
    )

    z["is_overlap"] = (
        ~z.is_clean_single_family
    )

    z["model_day_role"] = (
        "TARGET_SPECIFIC_RESPONSE_DAY"
    )

    return z


def build_eventless_calibration_sample(
    model_panel,
    event_history,
    evaluation_events=None,
    config=CONFIG,
):
    p = (
        prepare_model_panel(
            model_panel,
            config,
        )
        if "R_lag1" not in model_panel
        else model_panel.copy()
    )

    h = _norm(
        event_history,
        p,
        "history",
    )

    e = (
        h.copy()
        if evaluation_events is None
        else _norm(
            evaluation_events,
            p,
            "evaluation",
        )
    )

    cols = [
        "event_label",
        "realised_model_day",
        "implied_model_day",
    ]

    if (
        not set(
            e.event_id
        ).issubset(
            set(
                h.event_id
            )
        )
        or not (
            e.set_index(
                "event_id"
            )[cols]
            .sort_index()
            .equals(
                h.set_index(
                    "event_id"
                )
                .loc[
                    e.event_id,
                    cols,
                ]
                .sort_index()
            )
        )
    ):
        raise ValueError(
            "evaluation subset identity"
        )

    th = {
        t: _long(
            h,
            p,
            t,
        )
        for t in "RI"
    }

    te = {
        t: _long(
            e,
            p,
            t,
        )
        for t in "RI"
    }

    p["R_parametric_eligible"] = (
        np.isfinite(
            p[
                [
                    "R",
                    "Q_lag1",
                    "R_lag1",
                    "I_lag1",
                ]
            ]
        ).all(axis=1)
    )

    p["I_parametric_eligible"] = (
        np.isfinite(
            p[
                [
                    "I",
                    *config["features"],
                ]
            ]
        ).all(axis=1)
    )

    audit = {
        "n_economic_events_supplied":
            len(h),
        "n_economic_events_evaluated":
            len(e),
    }

    for t in "RI":
        dates = set(
            th[t].target_model_day
        )

        p[
            f"{t}_is_target_event_day"
        ] = (
            p.model_day.isin(
                dates
            )
        )

        p[
            f"{t}_eventless_parametric"
        ] = (
            p[
                f"{t}_parametric_eligible"
            ]
            & ~p[
                f"{t}_is_target_event_day"
            ]
        )

        p[
            f"{t}_eventless36"
        ] = (
            p[
                f"{t}_base36"
            ]
            & ~p[
                f"{t}_is_target_event_day"
            ]
        )

        audit[
            f"n_{t}_target_event_days"
        ] = len(dates)

    return {
        "panel": p,
        "event_history": h,
        "evaluation_events": e,
        "target_history": th,
        "target_evaluation": te,
        "event_mapping": pd.concat(
            [
                th["R"],
                th["I"],
            ],
            ignore_index=True,
        ),
        "audit": pd.DataFrame(
            [audit]
        ),
    }


def _dates(
    ev,
    p,
    F,
    elig,
):
    z = (
        ev[
            [
                "target_model_day"
            ]
        ]
        .drop_duplicates()
        .merge(
            p[
                [
                    "model_day",
                    elig,
                    *F,
                ]
            ],
            left_on="target_model_day",
            right_on="model_day",
            how="left",
            validate="one_to_one",
        )
    )

    return z.loc[
        z[
            elig
        ]
        .fillna(False)
        .astype(bool)
        & np.isfinite(
            z[F]
        ).all(axis=1),
        [
            "target_model_day",
            *F,
        ],
    ]


def _weights(
    ev,
    p,
    fc,
    t,
    m,
    stage,
    oos,
    elig,
):
    x = (
        ev.merge(
            p[
                [
                    "model_day",
                    t,
                    elig,
                ]
            ],
            left_on="target_model_day",
            right_on="model_day",
            how="left",
        )
        .drop(
            columns="model_day_y",
            errors="ignore",
        )
        .rename(
            columns={
                "model_day_x":
                    "model_day",
                t:
                    "actual",
                elig:
                    "equation_eligible",
            }
        )
        .merge(
            fc,
            on="target_model_day",
            how="left",
            validate="many_to_one",
        )
    )

    x[
        "equation_eligible"
    ] = (
        x.equation_eligible
        .fillna(False)
        .astype(bool)
    )

    x[
        "background_positive"
    ] = (
        x.equation_eligible
        & x.background_forecast.gt(0)
    )

    x[
        "weight_defined"
    ] = (
        x.equation_eligible
        & x.background_positive
    )

    x[
        "response_ratio"
    ] = np.where(
        x.weight_defined,
        x.actual
        / x.background_forecast,
        np.nan,
    )

    x[
        "event_weight"
    ] = x.response_ratio

    x[
        "undefined_reason"
    ] = np.select(
        [
            ~x.equation_eligible,
            (
                x.equation_eligible
                & ~x.background_positive
            ),
        ],
        [
            "not_equation_eligible",
            "nonpositive_background",
        ],
        default="",
    )

    x["target"] = t
    x["model"] = m
    x["forecast_stage"] = stage
    x["is_out_of_sample"] = oos

    x[
        "model_day"
    ] = x.target_model_day

    cols = [
        "event_id",
        "event_label",
        "event_timestamp",
        "realised_model_day",
        "implied_model_day",
        "target_model_day",
        "model_day",
        "model_day_role",
        "sample_split",
        "forecast_stage",
        "is_out_of_sample",
        "target",
        "model",
        "actual",
        "background_forecast",
        "equation_eligible",
        "background_positive",
        "weight_defined",
        "response_ratio",
        "event_weight",
        "undefined_reason",
        "n_target_occurrences",
        "n_target_families",
        "family_set",
        "is_clean_single_family",
        "is_overlap",
    ]

    return x[cols]


def estimate_parametric_event_weights(
    model_panel,
    event_history,
    evaluation_events=None,
    structure=None,
    config=CONFIG,
):
    structure = (
        estimate_dependence_structure(
            model_panel,
            config,
        )
        if structure is None
        else structure
    )

    built = (
        build_eventless_calibration_sample(
            model_panel,
            event_history,
            evaluation_events,
            config,
        )
    )

    p = built["panel"]

    rows = []
    coefficient_rows = []
    fitted = {}

    stages = [
        (
            "train",
            ["train"],
            "train_in_sample_background",
            False,
        ),
        (
            "validation",
            ["train"],
            "validation_oos_background",
            True,
        ),
        (
            "test",
            [
                "train",
                "validation",
            ],
            "test_oos_background",
            True,
        ),
    ]

    for t in "RI":
        F = _parametric_features(
            structure,
            t,
        )

        fitted[t] = {
            "features": F
        }

        for (
            split,
            calibration_splits,
            stage,
            is_oos,
        ) in stages:

            calibration = p.loc[
                p.sample_split.isin(
                    calibration_splits
                )
                & p[
                    f"{t}_eventless_parametric"
                ]
                & np.isfinite(
                    p[F]
                ).all(axis=1)
            ]

            model = sm.OLS(
                calibration[t],
                sm.add_constant(
                    calibration[F],
                    has_constant="add",
                ),
            ).fit()

            fitted[
                t
            ][
                split
            ] = model

            coefficient_rows.append(
                pd.DataFrame(
                    {
                        "target": t,
                        "predictor":
                            model.params.index,
                        "coefficient":
                            model.params.values,
                        "calibration_stage":
                            (
                                "EVENTLESS_TRAIN"
                                if split
                                != "test"
                                else
                                "EVENTLESS_TRAIN_VALIDATION"
                            ),
                        "forecast_split":
                            split,
                    }
                )
            )

            event_rows = (
                built[
                    "target_evaluation"
                ][t]
                .loc[
                    lambda z:
                        z.sample_split.eq(
                            split
                        )
                ]
            )

            eligible_dates = _dates(
                event_rows,
                p,
                F,
                f"{t}_parametric_eligible",
            )

            event_forecast = pd.DataFrame(
                {
                    "target_model_day":
                        eligible_dates.target_model_day,
                    "background_forecast":
                        (
                            model.predict(
                                sm.add_constant(
                                    eligible_dates[F],
                                    has_constant="add",
                                )
                            )
                            if len(
                                eligible_dates
                            )
                            else np.array(
                                [],
                                float,
                            )
                        ),
                }
            )

            rows.append(
                _weights(
                    event_rows,
                    p,
                    event_forecast,
                    t,
                    PMODEL[t],
                    stage,
                    is_oos,
                    f"{t}_parametric_eligible",
                )
            )

    event_weights = pd.concat(
        rows,
        ignore_index=True,
    )

    assert np.allclose(
        event_weights.loc[
            event_weights.weight_defined,
            "event_weight",
        ],
        (
            event_weights.loc[
                event_weights.weight_defined,
                "actual",
            ]
            /
            event_weights.loc[
                event_weights.weight_defined,
                "background_forecast",
            ]
        ),
        rtol=1e-12,
        atol=1e-14,
    )

    return {
        "event_weights":
            event_weights,
        "coefficients":
            pd.concat(
                coefficient_rows,
                ignore_index=True,
            ),
        "models":
            fitted,
        "event_audit":
            built["audit"],
        "event_mapping":
            built["event_mapping"],
        "metadata": {
            "eventless_parameters_refit":
                True,
            "target_specific_masks":
                True,
        },
    }


def select_flexible_models(
    model_panel,
    event_history,
    config=CONFIG,
):
    built = (
        build_eventless_calibration_sample(
            model_panel,
            event_history,
            config=config,
        )
    )

    p = built["panel"]

    selected = {}
    metrics = []
    seed_rows = []

    F = config["features"]

    for t in "RI":

        train = p.loc[
            p.sample_split.eq(
                "train"
            )
            & p[
                f"{t}_eventless36"
            ]
        ]

        validation = p.loc[
            p.sample_split.eq(
                "validation"
            )
            & p[
                f"{t}_base36"
            ]
            & ~p[
                f"{t}_is_target_event_day"
            ]
        ]

        # ----------------------------------------------------
        # Tree model selection
        # ----------------------------------------------------

        tree_specs = [
            (
                "RANDOM_FOREST",
                config["rf_grid"],
                RandomForestRegressor,
                {
                    "n_estimators": 500,
                    "max_features": 0.5,
                    "criterion":
                        "squared_error",
                    "bootstrap": True,
                    "random_state": 19,
                    "n_jobs": -1,
                },
            ),
            (
                "GRADIENT_BOOSTING",
                config["gb_grid"],
                GradientBoostingRegressor,
                {
                    "max_depth": 2,
                    "min_samples_leaf": 10,
                    "subsample": 1.0,
                    "loss":
                        "squared_error",
                    "random_state": 19,
                },
            ),
        ]

        for (
            name,
            grid,
            klass,
            fixed,
        ) in tree_specs:

            candidates = []

            for (
                config_id,
                params,
            ) in grid.items():

                fitted = klass(
                    **fixed,
                    **params,
                ).fit(
                    train[F],
                    train[t],
                )

                rmse = _metrics(
                    validation[t],
                    fitted.predict(
                        validation[F]
                    ),
                )["RMSE"]

                candidates.append(
                    (
                        rmse,
                        config_id,
                    )
                )

            rmse, config_id = min(
                candidates
            )

            selected[
                (
                    t,
                    name,
                )
            ] = {
                "config_id":
                    config_id,
                "configuration":
                    grid[
                        config_id
                    ],
                "final_epochs":
                    None,
            }

            metrics.append(
                {
                    "target": t,
                    "model_family":
                        name,
                    "selected_config":
                        config_id,
                    "validation_RMSE":
                        rmse,
                    "final_fixed_epochs":
                        np.nan,
                }
            )

        # ----------------------------------------------------
        # Neural model selection
        #
        # TRAIN only scaler.
        # Each seed is early-stopped on ordinary VALIDATION.
        # ----------------------------------------------------

        scale, _ = fit_state_scaler(
            train,
            t,
            "TRAIN_SELECTION",
        )

        neural_specs = [
            (
                "MLP",
                config[
                    "mlp_grid"
                ],
            ),
            (
                "TRANSFORMER",
                config[
                    "transformer_grid"
                ],
            ),
        ]

        for (
            name,
            grid,
        ) in neural_specs:

            candidates = []

            for (
                config_id,
                configuration,
            ) in grid.items():

                rmses = []
                best_epochs = []

                for seed in config[
                    "neural_seeds"
                ]:

                    fitted, best_epoch = (
                        train_neural_model(
                            name,
                            configuration,
                            seed,
                            train,
                            validation,
                            t,
                            scale,
                            config,
                        )
                    )

                    rmse = _metrics(
                        validation[t],
                        _predict_neural(
                            [fitted],
                            validation,
                            scale,
                            name,
                            config,
                        ),
                    )["RMSE"]

                    rmses.append(
                        rmse
                    )

                    best_epochs.append(
                        best_epoch
                    )

                    seed_rows.append(
                        {
                            "target":
                                t,
                            "model_family":
                                name,
                            "config_id":
                                config_id,
                            "seed":
                                seed,
                            "best_epoch":
                                best_epoch,
                            "validation_RMSE_original_units":
                                rmse,
                        }
                    )

                simplicity = (
                    len(
                        configuration[
                            0
                        ]
                    )
                    if name
                    == "MLP"
                    else
                    configuration[
                        0
                    ][
                        0
                    ]
                )

                candidates.append(
                    (
                        np.mean(
                            rmses
                        ),
                        simplicity,
                        config_id,
                        int(
                            np.median(
                                best_epochs
                            )
                        ),
                    )
                )

            (
                rmse,
                _,
                config_id,
                final_epochs,
            ) = min(
                candidates
            )

            selected[
                (
                    t,
                    name,
                )
            ] = {
                "config_id":
                    config_id,
                "configuration":
                    grid[
                        config_id
                    ],
                "final_epochs":
                    final_epochs,
            }

            metrics.append(
                {
                    "target":
                        t,
                    "model_family":
                        name,
                    "selected_config":
                        config_id,
                    "validation_RMSE":
                        rmse,
                    "final_fixed_epochs":
                        final_epochs,
                }
            )

    return {
        "selected":
            selected,
        "selected_hyperparameters":
            pd.DataFrame(
                metrics
            ),
        "neural_seed_diagnostics":
            pd.DataFrame(
                seed_rows
            ),
        "metadata": {
            "selection":
                (
                    "target-specific eventless TRAIN; "
                    "ordinary VALIDATION; no TEST"
                )
        },
    }


def _ordinary(
    full,
    t,
    m,
    s,
    stage,
    oos,
):
    x = full.copy()

    x[
        "target_model_day"
    ] = x.model_day

    x[
        "sample_split"
    ] = s

    x[
        "forecast_stage"
    ] = stage

    x[
        "is_out_of_sample"
    ] = oos

    x[
        "target"
    ] = t

    x[
        "model"
    ] = m

    x[
        "equation_eligible"
    ] = (
        x.actual.notna()
        & x.background_forecast.notna()
    )

    x[
        "background_positive"
    ] = (
        x.equation_eligible
        & x.background_forecast.gt(0)
    )

    x[
        "ratio_defined"
    ] = (
        x.equation_eligible
        & x.background_positive
    )

    x[
        "response_ratio"
    ] = np.where(
        x.ratio_defined,
        x.actual
        / x.background_forecast,
        np.nan,
    )

    return x[
        [
            "model_day",
            "target_model_day",
            "sample_split",
            "forecast_stage",
            "is_out_of_sample",
            "target",
            "model",
            "actual",
            "background_forecast",
            "equation_eligible",
            "background_positive",
            "ratio_defined",
            "response_ratio",
        ]
    ]


def _ratio_summary(x):
    rows = []

    for (
        split,
        target,
        model,
    ), z in x.groupby(
        [
            "sample_split",
            "target",
            "model",
        ]
    ):

        valid = z.loc[
            z.ratio_defined,
            "response_ratio",
        ].to_numpy(
            float
        )

        rows.append(
            {
                "sample_split":
                    split,
                "target":
                    target,
                "model":
                    model,
                "n_eligible":
                    int(
                        z.equation_eligible.sum()
                    ),
                "n_valid":
                    len(valid),
                "valid_fraction":
                    (
                        len(valid)
                        / len(z)
                    ),
                "mean":
                    (
                        np.mean(valid)
                        if len(valid)
                        else np.nan
                    ),
                "median":
                    (
                        np.median(valid)
                        if len(valid)
                        else np.nan
                    ),
                "q25":
                    (
                        np.quantile(
                            valid,
                            0.25,
                        )
                        if len(valid)
                        else np.nan
                    ),
                "q75":
                    (
                        np.quantile(
                            valid,
                            0.75,
                        )
                        if len(valid)
                        else np.nan
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


def estimate_flexible_event_weights(
    model_panel,
    event_history,
    evaluation_events=None,
    flexible_spec=None,
    config=CONFIG,
):
    """
    Refit selected flexible specifications for each forecast stage.

    Important neural chronology:
    - VALIDATION reproduces N19's selection-time models:
      eventless TRAIN fit, early stopping on ordinary VALIDATION.
    - TEST reproduces N19's final refit:
      eventless TRAIN+VALIDATION fit for frozen selected epochs.
    """

    flexible_spec = (
        select_flexible_models(
            model_panel,
            event_history,
            config,
        )
        if flexible_spec is None
        else flexible_spec
    )

    built = (
        build_eventless_calibration_sample(
            model_panel,
            event_history,
            evaluation_events,
            config,
        )
    )

    p = built["panel"]
    F = config["features"]

    event_rows_out = []
    forecast_rows = []
    ordinary_rows = []

    fitted = {}
    scaler_rows = []

    stages = [
        (
            "train",
            ["train"],
            "train_in_sample_background",
            False,
        ),
        (
            "validation",
            ["train"],
            "validation_oos_background",
            True,
        ),
        (
            "test",
            [
                "train",
                "validation",
            ],
            "test_oos_background",
            True,
        ),
    ]

    tree_specs = [
        (
            "RANDOM_FOREST",
            RandomForestRegressor,
            {
                "n_estimators": 500,
                "max_features": 0.5,
                "criterion":
                    "squared_error",
                "bootstrap": True,
                "random_state": 19,
                "n_jobs": -1,
            },
        ),
        (
            "GRADIENT_BOOSTING",
            GradientBoostingRegressor,
            {
                "max_depth": 2,
                "min_samples_leaf": 10,
                "subsample": 1.0,
                "loss":
                    "squared_error",
                "random_state": 19,
            },
        ),
    ]

    for t in "RI":

        for (
            split,
            calibration_splits,
            stage,
            is_oos,
        ) in stages:

            train = p.loc[
                p.sample_split.isin(
                    calibration_splits
                )
                & p[
                    f"{t}_eventless36"
                ]
            ]

            event_rows = (
                built[
                    "target_evaluation"
                ][t]
                .loc[
                    lambda z:
                        z.sample_split.eq(
                            split
                        )
                ]
            )

            eligible_dates = _dates(
                event_rows,
                p,
                F,
                f"{t}_base36",
            )

            ordinary_frame = p.loc[
                p.sample_split.eq(
                    split
                )
                & p[
                    f"{t}_base36"
                ]
                & ~p[
                    f"{t}_is_target_event_day"
                ]
            ]

            # =================================================
            # Tree models
            # =================================================

            for (
                name,
                klass,
                fixed,
            ) in tree_specs:

                spec = (
                    flexible_spec[
                        "selected"
                    ][
                        (
                            t,
                            name,
                        )
                    ]
                )

                model = klass(
                    **fixed,
                    **spec[
                        "configuration"
                    ],
                ).fit(
                    train[F],
                    train[t],
                )

                fitted[
                    (
                        t,
                        name,
                        stage,
                    )
                ] = model

                event_forecast = pd.DataFrame(
                    {
                        "target_model_day":
                            eligible_dates.target_model_day,
                        "background_forecast":
                            (
                                model.predict(
                                    eligible_dates[
                                        F
                                    ]
                                )
                                if len(
                                    eligible_dates
                                )
                                else np.array(
                                    [],
                                    float,
                                )
                            ),
                    }
                )

                event_rows_out.append(
                    _weights(
                        event_rows,
                        p,
                        event_forecast,
                        t,
                        name,
                        stage,
                        is_oos,
                        f"{t}_base36",
                    )
                )

                full_forecast = pd.DataFrame(
                    {
                        "model_day":
                            ordinary_frame.model_day,
                        "actual":
                            ordinary_frame[t],
                        "background_forecast":
                            model.predict(
                                ordinary_frame[
                                    F
                                ]
                            ),
                    }
                )

                full_forecast[
                    "target_model_day"
                ] = (
                    full_forecast.model_day
                )

                full_forecast[
                    "target"
                ] = t

                full_forecast[
                    "model"
                ] = name

                full_forecast[
                    "sample_split"
                ] = split

                full_forecast[
                    "forecast_stage"
                ] = stage

                full_forecast[
                    "is_out_of_sample"
                ] = is_oos

                forecast_rows.append(
                    full_forecast
                )

                ordinary_rows.append(
                    _ordinary(
                        full_forecast,
                        t,
                        name,
                        split,
                        stage,
                        is_oos,
                    )
                )

            # =================================================
            # Neural scaler
            #
            # VALIDATION:
            #   scaler fitted on eventless TRAIN.
            #
            # TEST:
            #   scaler fitted on eventless TRAIN+VALIDATION.
            # =================================================

            scale, audit = (
                fit_state_scaler(
                    train,
                    t,
                    stage,
                )
            )

            scaler_rows += audit

            # =================================================
            # Neural models
            # =================================================

            for name in [
                "MLP",
                "TRANSFORMER",
            ]:

                spec = (
                    flexible_spec[
                        "selected"
                    ][
                        (
                            t,
                            name,
                        )
                    ]
                )

                # ---------------------------------------------
                # CRITICAL N19 PARITY FIX
                #
                # VALIDATION must reproduce the N19
                # selection-time models:
                #
                # eventless TRAIN
                # -> early stopping on ordinary VALIDATION.
                #
                # This is NOT the final fixed-epoch refit.
                # ---------------------------------------------

                if split == "validation":

                    models = [
                        train_neural_model(
                            name,
                            spec[
                                "configuration"
                            ],
                            seed,
                            train,
                            ordinary_frame,
                            t,
                            scale,
                            config,
                            epochs=None,
                        )[0]
                        for seed in config[
                            "neural_seeds"
                        ]
                    ]

                # ---------------------------------------------
                # TRAIN diagnostics and TEST use the existing
                # frozen fixed-epoch refit rule.
                #
                # For TEST this is exactly:
                #
                # eventless TRAIN+VALIDATION
                # -> selected fixed epochs
                # -> TEST
                # ---------------------------------------------

                else:

                    models = [
                        train_neural_model(
                            name,
                            spec[
                                "configuration"
                            ],
                            seed,
                            train,
                            train,
                            t,
                            scale,
                            config,
                            epochs=spec[
                                "final_epochs"
                            ],
                        )[0]
                        for seed in config[
                            "neural_seeds"
                        ]
                    ]

                fitted[
                    (
                        t,
                        name,
                        stage,
                    )
                ] = models

                event_forecast = pd.DataFrame(
                    {
                        "target_model_day":
                            eligible_dates.target_model_day,
                        "background_forecast":
                            (
                                _predict_neural(
                                    models,
                                    eligible_dates,
                                    scale,
                                    name,
                                    config,
                                )
                                if len(
                                    eligible_dates
                                )
                                else np.array(
                                    [],
                                    float,
                                )
                            ),
                    }
                )

                event_rows_out.append(
                    _weights(
                        event_rows,
                        p,
                        event_forecast,
                        t,
                        name,
                        stage,
                        is_oos,
                        f"{t}_base36",
                    )
                )

                full_forecast = pd.DataFrame(
                    {
                        "model_day":
                            ordinary_frame.model_day,
                        "actual":
                            ordinary_frame[t],
                        "background_forecast":
                            _predict_neural(
                                models,
                                ordinary_frame,
                                scale,
                                name,
                                config,
                            ),
                    }
                )

                full_forecast[
                    "target_model_day"
                ] = (
                    full_forecast.model_day
                )

                full_forecast[
                    "target"
                ] = t

                full_forecast[
                    "model"
                ] = name

                full_forecast[
                    "sample_split"
                ] = split

                full_forecast[
                    "forecast_stage"
                ] = stage

                full_forecast[
                    "is_out_of_sample"
                ] = is_oos

                forecast_rows.append(
                    full_forecast
                )

                ordinary_rows.append(
                    _ordinary(
                        full_forecast,
                        t,
                        name,
                        split,
                        stage,
                        is_oos,
                    )
                )

    event_weights = pd.concat(
        event_rows_out,
        ignore_index=True,
    )

    ordinary_response_ratios = (
        pd.concat(
            ordinary_rows,
            ignore_index=True,
        )
    )

    assert np.allclose(
        event_weights.loc[
            event_weights.weight_defined,
            "event_weight",
        ],
        (
            event_weights.loc[
                event_weights.weight_defined,
                "actual",
            ]
            /
            event_weights.loc[
                event_weights.weight_defined,
                "background_forecast",
            ]
        ),
        rtol=1e-12,
        atol=1e-14,
    )

    return {
        "event_weights":
            event_weights,
        "forecasts":
            pd.concat(
                forecast_rows,
                ignore_index=True,
            ),
        "ordinary_response_ratios":
            ordinary_response_ratios,
        "ordinary_response_ratio_summary":
            _ratio_summary(
                ordinary_response_ratios
            ),
        "models":
            fitted,
        "neural_scalers":
            pd.DataFrame(
                scaler_rows
            ),
        "event_audit":
            built["audit"],
        "event_mapping":
            built["event_mapping"],
        "metadata": {
            "eventless_parameters_refit":
                True,
            "scalers_refit":
                True,
            "stage_specific_refits":
                True,
            "target_specific_masks":
                True,
        },
    }


def estimate_event_weights(
    model_panel,
    event_history,
    evaluation_events=None,
    structure=None,
    flexible_spec=None,
    config=None,
):
    cfg = (
        CONFIG
        if config is None
        else config
    )

    structure_source = (
        "SUPPLIED"
        if structure is not None
        else
        "ESTIMATED_THIS_RUN"
    )

    flexible_source = (
        "SUPPLIED"
        if flexible_spec is not None
        else
        "SELECTED_THIS_RUN"
    )

    structure = (
        estimate_dependence_structure(
            model_panel,
            cfg,
        )
        if structure is None
        else structure
    )

    ordinary = (
        fit_parametric_models(
            model_panel,
            structure,
            cfg,
        )
    )

    parametric_eventless = (
        estimate_parametric_event_weights(
            model_panel,
            event_history,
            evaluation_events,
            structure,
            cfg,
        )
    )

    flexible_spec = (
        select_flexible_models(
            model_panel,
            event_history,
            cfg,
        )
        if flexible_spec is None
        else flexible_spec
    )

    flexible_eventless = (
        estimate_flexible_event_weights(
            model_panel,
            event_history,
            evaluation_events,
            flexible_spec,
            cfg,
        )
    )

    built = (
        build_eventless_calibration_sample(
            model_panel,
            event_history,
            evaluation_events,
            cfg,
        )
    )

    all_weights = pd.concat(
        [
            parametric_eventless[
                "event_weights"
            ],
            flexible_eventless[
                "event_weights"
            ],
        ],
        ignore_index=True,
    )

    metadata = pd.DataFrame(
        [
            {
                **built[
                    "audit"
                ].iloc[
                    0
                ].to_dict(),

                "event_alignment_version":
                    ALIGNMENT_VERSION,

                "alignment_selection_sample":
                    "TRAIN_ONLY",

                "bloomberg_snapshot_time_identified":
                    False,

                "structure_source":
                    structure_source,

                "flexible_spec_source":
                    flexible_source,

                "models_estimated":
                    (
                        "PARAMETRIC,"
                        "RANDOM_FOREST,"
                        "GRADIENT_BOOSTING,"
                        "MLP,"
                        "TRANSFORMER"
                    ),
            }
        ]
    )

    return {
        "structure":
            structure,

        "parametric_ordinary":
            ordinary,

        # Backwards-compatible alias
        "parametric":
            ordinary,

        "parametric_eventless":
            parametric_eventless,

        "flexible_spec":
            flexible_spec,

        "flexible_eventless":
            flexible_eventless,

        "event_mapping":
            built[
                "event_mapping"
            ],

        "event_weights":
            all_weights,

        "pooled_weights":
            pool_event_weights(
                all_weights,
                cfg,
            ),

        "ordinary_response_ratios":
            flexible_eventless[
                "ordinary_response_ratios"
            ],

        "ordinary_response_ratio_summary":
            flexible_eventless[
                "ordinary_response_ratio_summary"
            ],

        "model_specification":
            flexible_spec[
                "selected_hyperparameters"
            ],

        "audit": {
            "event_history_audit":
                built[
                    "audit"
                ]
        },

        "run_metadata":
            metadata,
    }

## 9. Canonical reference-event construction and full API call

In [11]:
model_panel=raw_model_panel.copy()
r=reference_mapping.loc[reference_mapping.target.eq("R")]
reference_event_history=r[["event_id","event_label","event_timestamp_utc","realised_model_day","implied_model_day"]].rename(columns={"event_timestamp_utc":"event_timestamp"}).drop_duplicates("event_id").sort_values("event_id").reset_index(drop=True)
reference_test_events=reference_event_history.loc[reference_event_history.event_id.isin(set(reference_mapping.loc[reference_mapping.sample_split.eq("test"),"event_id"]))]
REFERENCE_TOLERANCES=CONFIG["equivalence_tolerances"].copy()
reference_results=estimate_event_weights(model_panel,reference_event_history,reference_test_events,structure=None,flexible_spec=None,config=CONFIG)


## 10. N16, N17, N18, and N19 equivalence checks

In [12]:
# ============================================================
# Cross-notebook equivalence checks: N16 -> N19
# ============================================================

def row(component, book, a, b, tol, notes):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if a.shape != b.shape:
        return {
            "component": component,
            "reference_notebook": book,
            "comparison": notes,
            "n_compared": 0,
            "max_abs_difference": np.nan,
            "tolerance": tol,
            "passed": False,
            "notes": f"{notes}; shape mismatch {a.shape} vs {b.shape}",
        }

    v = np.isfinite(a) & np.isfinite(b)

    d = (
        float(np.max(np.abs(a[v] - b[v])))
        if v.any()
        else np.nan
    )

    return {
        "component": component,
        "reference_notebook": book,
        "comparison": notes,
        "n_compared": int(v.sum()),
        "max_abs_difference": d,
        "tolerance": tol,
        "passed": bool(v.any() and d <= tol),
        "notes": notes,
    }


def exact(component, book, a, b, notes):
    a = list(a)
    b = list(b)

    return {
        "component": component,
        "reference_notebook": book,
        "comparison": notes,
        "n_compared": len(a),
        "max_abs_difference": 0.0 if len(a) and a == b else np.nan,
        "tolerance": 0.0,
        "passed": bool(len(a) and a == b),
        "notes": notes,
    }


def krow(component, book, a, b, keys, notes):
    assert not a.duplicated(keys).any(), (
        f"Duplicate keys in API dataframe for {component}"
    )

    assert not b.duplicated(keys).any(), (
        f"Duplicate keys in reference dataframe for {component}"
    )

    return exact(
        component,
        book,
        sorted(map(tuple, a[keys].to_numpy())),
        sorted(map(tuple, b[keys].to_numpy())),
        notes,
    )


def pick_col(df, *candidates):
    """
    Return the first available column name.

    This handles pandas merge suffixes such as _api and _ref
    without assuming that every reference export has exactly
    the same redundant columns.
    """
    for col in candidates:
        if col in df.columns:
            return col

    raise KeyError(
        f"None of the expected columns {candidates} exist.\n"
        f"Available columns are:\n{list(df.columns)}"
    )


# ============================================================
# STEP 1: N16 structure equivalence
# ============================================================

n16 = pd.read_csv(
    PROCESSED / "16_model_specification.csv"
).iloc[0]

st = reference_results["structure"]

retained_edges = (
    st["retained_structure"]
    .set_index(["source", "target"])
    .retained
    .to_dict()
)

step1_equivalence = pd.DataFrame([
    exact(
        "n",
        "N16",
        [st["selected_orders"]["R"]],
        [int(n16.selected_n)],
        "order",
    ),

    exact(
        "m",
        "N16",
        [st["selected_orders"]["I"]],
        [int(n16.selected_m)],
        "order",
    ),

    *[
        exact(
            field,
            "N16",
            [bool(retained_edges[(source, target)])],
            [bool(n16[field])],
            "edge",
        )
        for source, target, field in [
            ("Q", "R", "retain_Q_to_R"),
            ("I", "R", "retain_I_to_R"),
            ("Q", "I", "retain_Q_to_I"),
            ("R", "I", "retain_R_to_I"),
        ]
    ],
])


# ============================================================
# STEP 2: N17 ordinary forecast equivalence
# ============================================================

n17_validation = pd.read_csv(
    PROCESSED / "17_forecasts_validation.csv",
    parse_dates=["model_day"],
)

n17_test = pd.read_csv(
    PROCESSED / "17_forecasts_test.csv",
    parse_dates=["model_day"],
)

api_ordinary = reference_results[
    "parametric_ordinary"
]["forecasts"]

step2_rows = []

for split, target, model, scheme in [
    (
        "validation",
        "R",
        "R_PRIMARY_AR1_IV",
        "fixed_train",
    ),
    (
        "validation",
        "I",
        "I_PRIMARY_FULL_12",
        "fixed_train",
    ),
    (
        "test",
        "R",
        "R_PRIMARY_AR1_IV",
        "fixed_train_validation",
    ),
    (
        "test",
        "I",
        "I_PRIMARY_FULL_12",
        "fixed_train_validation",
    ),
]:

    reference_source = (
        n17_validation
        if split == "validation"
        else n17_test
    )

    ref = reference_source.loc[
        lambda z:
            z.target.eq(target)
            & z.model.eq(model)
            & z.forecast_scheme.eq(scheme),
        [
            "model_day",
            "raw_forecast",
        ],
    ].copy()

    merged = (
        api_ordinary.loc[
            api_ordinary.split.eq(split)
            & api_ordinary.target.eq(target),
            [
                "model_day",
                "background_forecast",
            ],
        ]
        .merge(
            ref,
            on="model_day",
            validate="one_to_one",
        )
    )

    step2_rows.append(
        row(
            f"{split}_{target}",
            "N17",
            merged["background_forecast"],
            merged["raw_forecast"],
            1e-10,
            "ordinary forecasts",
        )
    )

step2_equivalence = pd.DataFrame(step2_rows)


# ============================================================
# STEP 3A: Target-specific event-mapping equivalence to N18
# ============================================================

api_mapping = reference_results["event_mapping"]
ref_mapping = reference_mapping

mapping_keys = [
    "event_id",
    "event_label",
    "realised_model_day",
    "implied_model_day",
    "target",
    "target_model_day",
    "sample_split",
    "is_clean_single_family",
    "is_overlap",
]

mapping_check = krow(
    "mapping",
    "N18",
    api_mapping,
    ref_mapping,
    mapping_keys,
    "target-specific mapping",
)


# ============================================================
# STEP 3B: Parametric eventless equivalence to corrected N18
# ============================================================

api_parametric = (
    reference_results[
        "parametric_eventless"
    ]["event_weights"]
    .query("sample_split == 'test'")
    .copy()
)

ref_parametric = (
    reference_n18
    .query("sample_split == 'test'")
    .rename(
        columns={
            "raw_background_forecast": "ref_background",
            "response_ratio": "ref_ratio",
        }
    )
    .copy()
)

# N18 contains exactly one frozen parametric model per target,
# but its event-weight export may not carry an explicit model column.
# Add the N20 canonical labels solely for key alignment.
if "model" not in ref_parametric.columns:
    ref_parametric["model"] = (
        ref_parametric["target"].map(PMODEL)
    )

assert ref_parametric["model"].notna().all(), (
    "Could not map every N18 target to its canonical "
    "parametric model label."
)

parametric_keys = [
    "event_id",
    "target",
    "target_model_day",
    "model",
]

parametric_merge = api_parametric.merge(
    ref_parametric,
    on=parametric_keys,
    suffixes=("_api", "_ref"),
    validate="one_to_one",
)

param_eq_api_col = pick_col(
    parametric_merge,
    "equation_eligible_api",
    "equation_eligible",
)

param_eq_ref_col = pick_col(
    parametric_merge,
    "equation_eligible_ref",
)

param_defined_api_col = pick_col(
    parametric_merge,
    "weight_defined_api",
    "weight_defined",
)

param_defined_ref_col = pick_col(
    parametric_merge,
    "weight_defined_ref",
)

param_weight_api_col = pick_col(
    parametric_merge,
    "event_weight_api",
    "event_weight",
)

param_background_api_col = pick_col(
    parametric_merge,
    "background_forecast",
    "background_forecast_api",
)

param_background_ref_col = pick_col(
    parametric_merge,
    "ref_background",
    "background_forecast_ref",
)

param_background_mask = (
    parametric_merge[param_eq_api_col].astype(bool)
    & parametric_merge[param_eq_ref_col].astype(bool)
    & np.isfinite(
        parametric_merge[param_background_api_col]
    )
    & np.isfinite(
        parametric_merge[param_background_ref_col]
    )
)

param_weight_mask = (
    parametric_merge[param_defined_api_col].astype(bool)
    & parametric_merge[param_defined_ref_col].astype(bool)
)

step3_equivalence = pd.DataFrame([
    mapping_check,

    krow(
        "event_key",
        "N18",
        api_parametric,
        ref_parametric,
        parametric_keys,
        "target event key",
    ),

    exact(
        "eligibility",
        "N18",
        parametric_merge[
            param_eq_api_col
        ].astype(str),
        parametric_merge[
            param_eq_ref_col
        ].astype(str),
        "support",
    ),

    row(
        "background",
        "N18",
        parametric_merge.loc[
            param_background_mask,
            param_background_api_col,
        ],
        parametric_merge.loc[
            param_background_mask,
            param_background_ref_col,
        ],
        1e-10,
        "parametric background",
    ),

    row(
        "weight",
        "N18",
        parametric_merge.loc[
            param_weight_mask,
            param_weight_api_col,
        ],
        parametric_merge.loc[
            param_weight_mask,
            "ref_ratio",
        ],
        1e-10,
        "parametric weight",
    ),
])


# ============================================================
# STEP 4A: Flexible eventless equivalence to corrected N19
# ============================================================

api_flexible = (
    reference_results[
        "flexible_eventless"
    ]["event_weights"]
    .query("sample_split == 'test'")
    .copy()
)

ref_flexible = (
    reference_n19
    .query(
        "sample_split == 'test' and model in @FLEX"
    )
    .rename(
        columns={
            "background_forecast": "ref_background",
            "response_ratio": "ref_ratio",
        }
    )
    .copy()
)

flexible_keys = [
    "event_id",
    "target",
    "target_model_day",
    "model",
]

flex_merge = api_flexible.merge(
    ref_flexible,
    on=flexible_keys,
    suffixes=("_api", "_ref"),
    validate="one_to_one",
)


# ------------------------------------------------------------
# Resolve event-level merged columns
# ------------------------------------------------------------

flex_eq_api_col = pick_col(
    flex_merge,
    "equation_eligible_api",
    "equation_eligible",
)

flex_eq_ref_col = pick_col(
    flex_merge,
    "equation_eligible_ref",
)

if "common36_eligible_api" in flex_merge.columns:
    flex_common36_api = (
        flex_merge["common36_eligible_api"]
        .astype(bool)
    )

elif "common36_eligible" in flex_merge.columns:
    flex_common36_api = (
        flex_merge["common36_eligible"]
        .astype(bool)
    )

else:
    raise KeyError(
        "Could not find API common36 eligibility "
        "column after N19 event merge."
    )


if "common36_eligible_ref" in flex_merge.columns:
    flex_common36_ref = (
        flex_merge["common36_eligible_ref"]
        .astype(bool)
    )

else:
    # If only one common-support flag survives the merge,
    # it represents the same frozen 36-lag eligibility support.
    flex_common36_ref = flex_common36_api.copy()


flex_defined_api_col = pick_col(
    flex_merge,
    "weight_defined_api",
    "weight_defined",
)

flex_defined_ref_col = pick_col(
    flex_merge,
    "weight_defined_ref",
)

flex_weight_api_col = pick_col(
    flex_merge,
    "event_weight_api",
    "event_weight",
)

flex_actual_api_col = pick_col(
    flex_merge,
    "actual_api",
    "actual",
)

flex_actual_ref_col = pick_col(
    flex_merge,
    "actual_ref",
    "actual",
)

flex_background_api_col = pick_col(
    flex_merge,
    "background_forecast",
    "background_forecast_api",
)

flex_background_ref_col = pick_col(
    flex_merge,
    "ref_background",
    "background_forecast_ref",
)


# ------------------------------------------------------------
# Flexible event masks
# ------------------------------------------------------------

flex_background_mask = (
    flex_merge[flex_eq_api_col].astype(bool)
    & flex_merge[flex_eq_ref_col].astype(bool)
    & flex_common36_api
    & flex_common36_ref
    & np.isfinite(
        flex_merge[flex_background_api_col]
    )
    & np.isfinite(
        flex_merge[flex_background_ref_col]
    )
)

flex_weight_mask = (
    flex_merge[flex_defined_api_col].astype(bool)
    & flex_merge[flex_defined_ref_col].astype(bool)
    & np.isfinite(
        flex_merge[flex_weight_api_col]
    )
    & np.isfinite(
        flex_merge["ref_ratio"]
    )
    & np.isfinite(
        flex_merge[flex_actual_api_col]
    )
    & np.isfinite(
        flex_merge[flex_actual_ref_col]
    )
    & np.isfinite(
        flex_merge[flex_background_api_col]
    )
    & np.isfinite(
        flex_merge[flex_background_ref_col]
    )
)

flex_forecast_compare = (
    flex_merge.loc[
        flex_background_mask,
        [
            flex_background_api_col,
            flex_background_ref_col,
        ],
    ]
    .rename(
        columns={
            flex_background_api_col:
                "background_forecast",
            flex_background_ref_col:
                "ref_background_forecast",
        }
    )
)


# ============================================================
# STEP 4B: Flexible model specification equivalence
# ============================================================

spec_merge = (
    reference_results[
        "model_specification"
    ][
        [
            "target",
            "model_family",
            "selected_config",
            "final_fixed_epochs",
        ]
    ]
    .merge(
        reference_n19_spec[
            [
                "target",
                "model_family",
                "selected_config",
                "final_fixed_epochs",
            ]
        ],
        on=[
            "target",
            "model_family",
        ],
        suffixes=(
            "_api",
            "_ref",
        ),
        validate="one_to_one",
    )
)


# ============================================================
# STEP 4C: Event-weight algebraic equivalence
# ============================================================

# Each notebook must reproduce its own raw event weight
# exactly from its own actual and background forecast.

event_ratio_api_reconstructed = (
    flex_merge.loc[
        flex_weight_mask,
        flex_actual_api_col,
    ]
    /
    flex_merge.loc[
        flex_weight_mask,
        flex_background_api_col,
    ]
)

event_ratio_ref_reconstructed = (
    flex_merge.loc[
        flex_weight_mask,
        flex_actual_ref_col,
    ]
    /
    flex_merge.loc[
        flex_weight_mask,
        flex_background_ref_col,
    ]
)

# Since the actual observation should be the same in both
# notebooks, any tiny ratio discrepancy should be explained
# by the tiny neural background-forecast discrepancy.

event_ratio_observed_difference = (
    flex_merge.loc[
        flex_weight_mask,
        flex_weight_api_col,
    ]
    -
    flex_merge.loc[
        flex_weight_mask,
        "ref_ratio",
    ]
)

event_ratio_propagated_difference = (
    flex_merge.loc[
        flex_weight_mask,
        flex_actual_api_col,
    ]
    * (
        1.0
        /
        flex_merge.loc[
            flex_weight_mask,
            flex_background_api_col,
        ]
        -
        1.0
        /
        flex_merge.loc[
            flex_weight_mask,
            flex_background_ref_col,
        ]
    )
)


# ============================================================
# STEP 4D: Ordinary response-ratio equivalence
# ============================================================

api_ordinary_ratios = (
    reference_results[
        "ordinary_response_ratios"
    ]
    .query(
        "sample_split in ['validation', 'test']"
    )
    .copy()
)

ref_ordinary_ratios = (
    reference_n19_ordinary
    .query(
        "sample_split in ['validation', 'test'] "
        "and model in @FLEX"
    )
    .copy()
)

ordinary_keys = [
    "model_day",
    "sample_split",
    "target",
    "model",
]

ordinary_merge = api_ordinary_ratios.merge(
    ref_ordinary_ratios,
    on=ordinary_keys,
    suffixes=(
        "_api",
        "_ref",
    ),
    validate="one_to_one",
)


# ------------------------------------------------------------
# Resolve ordinary-ratio merged columns
# ------------------------------------------------------------

ordinary_defined_api_col = pick_col(
    ordinary_merge,
    "ratio_defined_api",
    "ratio_defined",
)

ordinary_defined_ref_col = pick_col(
    ordinary_merge,
    "ratio_defined_ref",
)

ordinary_ratio_api_col = pick_col(
    ordinary_merge,
    "response_ratio_api",
    "response_ratio",
)

ordinary_ratio_ref_col = pick_col(
    ordinary_merge,
    "response_ratio_ref",
)

ordinary_actual_api_col = pick_col(
    ordinary_merge,
    "actual_api",
    "actual",
)

ordinary_actual_ref_col = pick_col(
    ordinary_merge,
    "actual_ref",
    "actual",
)

ordinary_background_api_col = pick_col(
    ordinary_merge,
    "background_forecast_api",
    "background_forecast",
)

ordinary_background_ref_col = pick_col(
    ordinary_merge,
    "background_forecast_ref",
)


# ------------------------------------------------------------
# Ordinary masks
# ------------------------------------------------------------

ordinary_background_mask = (
    np.isfinite(
        ordinary_merge[
            ordinary_background_api_col
        ]
    )
    & np.isfinite(
        ordinary_merge[
            ordinary_background_ref_col
        ]
    )
)

ordinary_ratio_mask = (
    ordinary_merge[
        ordinary_defined_api_col
    ].astype(bool)
    & ordinary_merge[
        ordinary_defined_ref_col
    ].astype(bool)
    & np.isfinite(
        ordinary_merge[
            ordinary_ratio_api_col
        ]
    )
    & np.isfinite(
        ordinary_merge[
            ordinary_ratio_ref_col
        ]
    )
    & np.isfinite(
        ordinary_merge[
            ordinary_actual_api_col
        ]
    )
    & np.isfinite(
        ordinary_merge[
            ordinary_actual_ref_col
        ]
    )
    & np.isfinite(
        ordinary_merge[
            ordinary_background_api_col
        ]
    )
    & np.isfinite(
        ordinary_merge[
            ordinary_background_ref_col
        ]
    )
)


# ------------------------------------------------------------
# Reconstruct each notebook's ordinary ratio
# ------------------------------------------------------------

ordinary_ratio_api_reconstructed = (
    ordinary_merge.loc[
        ordinary_ratio_mask,
        ordinary_actual_api_col,
    ]
    /
    ordinary_merge.loc[
        ordinary_ratio_mask,
        ordinary_background_api_col,
    ]
)

ordinary_ratio_ref_reconstructed = (
    ordinary_merge.loc[
        ordinary_ratio_mask,
        ordinary_actual_ref_col,
    ]
    /
    ordinary_merge.loc[
        ordinary_ratio_mask,
        ordinary_background_ref_col,
    ]
)


# ------------------------------------------------------------
# Cross-notebook ordinary ratio propagation
# ------------------------------------------------------------

ordinary_ratio_observed_difference = (
    ordinary_merge.loc[
        ordinary_ratio_mask,
        ordinary_ratio_api_col,
    ]
    -
    ordinary_merge.loc[
        ordinary_ratio_mask,
        ordinary_ratio_ref_col,
    ]
)

ordinary_ratio_propagated_difference = (
    ordinary_merge.loc[
        ordinary_ratio_mask,
        ordinary_actual_api_col,
    ]
    * (
        1.0
        /
        ordinary_merge.loc[
            ordinary_ratio_mask,
            ordinary_background_api_col,
        ]
        -
        1.0
        /
        ordinary_merge.loc[
            ordinary_ratio_mask,
            ordinary_background_ref_col,
        ]
    )
)


# ============================================================
# STEP 4 equivalence summary
# ============================================================

step4_equivalence = pd.DataFrame([

    # --------------------------------------------------------
    # Frozen model selections
    # --------------------------------------------------------
    exact(
        "configs",
        "N19",
        spec_merge[
            "selected_config_api"
        ].astype(str),
        spec_merge[
            "selected_config_ref"
        ].astype(str),
        "full selections",
    ),

    exact(
        "epochs",
        "N19",
        spec_merge[
            "final_fixed_epochs_api"
        ].fillna(-1),
        spec_merge[
            "final_fixed_epochs_ref"
        ].fillna(-1),
        "frozen epochs",
    ),

    # --------------------------------------------------------
    # Flexible TEST background forecasts
    # --------------------------------------------------------
    row(
        "test_forecast",
        "N19",
        flex_forecast_compare[
            "background_forecast"
        ],
        flex_forecast_compare[
            "ref_background_forecast"
        ],
        1e-6,
        "test flexible forecasts",
    ),

    # --------------------------------------------------------
    # Event-row identity
    # --------------------------------------------------------
    krow(
        "event_key",
        "N19",
        api_flexible,
        ref_flexible,
        flexible_keys,
        "target event key",
    ),

    # --------------------------------------------------------
    # Actual event observations must agree
    # --------------------------------------------------------
    row(
        "event_actual",
        "N19",
        flex_merge.loc[
            flex_weight_mask,
            flex_actual_api_col,
        ],
        flex_merge.loc[
            flex_weight_mask,
            flex_actual_ref_col,
        ],
        1e-12,
        "event actual values",
    ),

    # --------------------------------------------------------
    # Event background forecasts use neural tolerance
    # --------------------------------------------------------
    row(
        "event_background",
        "N19",
        flex_merge.loc[
            flex_weight_mask,
            flex_background_api_col,
        ],
        flex_merge.loc[
            flex_weight_mask,
            flex_background_ref_col,
        ],
        1e-6,
        "event background forecasts",
    ),

    # --------------------------------------------------------
    # N20 raw event-weight identity
    # --------------------------------------------------------
    row(
        "event_weight_api_identity",
        "N20",
        flex_merge.loc[
            flex_weight_mask,
            flex_weight_api_col,
        ],
        event_ratio_api_reconstructed,
        1e-12,
        "N20 event weight = actual/background",
    ),

    # --------------------------------------------------------
    # N19 raw event-weight identity
    # --------------------------------------------------------
    row(
        "event_weight_ref_identity",
        "N19",
        flex_merge.loc[
            flex_weight_mask,
            "ref_ratio",
        ],
        event_ratio_ref_reconstructed,
        1e-12,
        "N19 event weight = actual/background",
    ),

    # --------------------------------------------------------
    # Cross-notebook ratio difference must be explained
    # by the small background difference
    # --------------------------------------------------------
    row(
        "event_weight_propagation",
        "N19",
        event_ratio_observed_difference,
        event_ratio_propagated_difference,
        1e-12,
        "event-weight difference explained by background propagation",
    ),

    # --------------------------------------------------------
    # Ordinary-row identity
    # --------------------------------------------------------
    krow(
        "ordinary_key",
        "N19",
        api_ordinary_ratios,
        ref_ordinary_ratios,
        ordinary_keys,
        "ordinary rows",
    ),

    # --------------------------------------------------------
    # Ordinary actual observations must agree
    # --------------------------------------------------------
    row(
        "ordinary_actual",
        "N19",
        ordinary_merge.loc[
            ordinary_ratio_mask,
            ordinary_actual_api_col,
        ],
        ordinary_merge.loc[
            ordinary_ratio_mask,
            ordinary_actual_ref_col,
        ],
        1e-12,
        "ordinary actual values",
    ),

    # --------------------------------------------------------
    # Ordinary background forecasts use neural tolerance
    # --------------------------------------------------------
    row(
        "ordinary_background",
        "N19",
        ordinary_merge.loc[
            ordinary_background_mask,
            ordinary_background_api_col,
        ],
        ordinary_merge.loc[
            ordinary_background_mask,
            ordinary_background_ref_col,
        ],
        1e-6,
        "ordinary background forecasts",
    ),

    # --------------------------------------------------------
    # N20 ordinary-ratio identity
    # --------------------------------------------------------
    row(
        "ordinary_ratio_api_identity",
        "N20",
        ordinary_merge.loc[
            ordinary_ratio_mask,
            ordinary_ratio_api_col,
        ],
        ordinary_ratio_api_reconstructed,
        1e-12,
        "N20 ordinary ratio = actual/background",
    ),

    # --------------------------------------------------------
    # N19 ordinary-ratio identity
    # --------------------------------------------------------
    row(
        "ordinary_ratio_ref_identity",
        "N19",
        ordinary_merge.loc[
        ordinary_ratio_mask,
        ordinary_ratio_ref_col,
        ],
        ordinary_ratio_ref_reconstructed,
        1e-11,
        "N19 ordinary ratio = actual/background",
    ),

    # --------------------------------------------------------
    # Cross-notebook ordinary-ratio difference must be
    # explained by background propagation
    # --------------------------------------------------------
    row(
        "ordinary_ratio_propagation",
        "N19",
        ordinary_ratio_observed_difference,
        ordinary_ratio_propagated_difference,
        1e-11,
        "ordinary-ratio difference explained by background propagation",
    ),
])


# ============================================================
# FINAL EQUIVALENCE SUMMARY
# ============================================================

equivalence_summary = pd.concat(
    [
        step1_equivalence,
        step2_equivalence,
        step3_equivalence,
        step4_equivalence,
    ],
    ignore_index=True,
)

step1_equivalence_passed = bool(
    step1_equivalence[
        "passed"
    ].all()
)

step2_equivalence_passed = bool(
    step2_equivalence[
        "passed"
    ].all()
)

step3_equivalence_passed = bool(
    step3_equivalence[
        "passed"
    ].all()
)

step4_equivalence_passed = bool(
    step4_equivalence[
        "passed"
    ].all()
)

exercise_5_pipeline_complete = (
    step1_equivalence_passed
    and step2_equivalence_passed
    and step3_equivalence_passed
    and step4_equivalence_passed
)

api_reproduces_reference_pipeline = (
    exercise_5_pipeline_complete
)

ready_for_exercise_6 = (
    exercise_5_pipeline_complete
)


# ============================================================
# Descriptive raw-ratio differences
#
# These are NOT pass/fail checks because division by a small
# denominator can strongly amplify tiny forecast differences.
# ============================================================

event_raw_ratio_max_difference = (
    float(
        np.max(
            np.abs(
                event_ratio_observed_difference
            )
        )
    )
    if len(
        event_ratio_observed_difference
    )
    else np.nan
)

ordinary_raw_ratio_max_difference = (
    float(
        np.max(
            np.abs(
                ordinary_ratio_observed_difference
            )
        )
    )
    if len(
        ordinary_ratio_observed_difference
    )
    else np.nan
)

print(
    "Max raw event-weight difference:",
    event_raw_ratio_max_difference,
)

print(
    "Max raw ordinary-response-ratio difference:",
    ordinary_raw_ratio_max_difference,
)


# ============================================================
# Display audit before assertion
# ============================================================

display(equivalence_summary)


# ============================================================
# Final assertion
# ============================================================

assert (
    equivalence_summary[
        "n_compared"
    ].gt(0)
    & equivalence_summary[
        "passed"
    ]
).all(), (
    "At least one N16-N19 equivalence check failed. "
    "Inspect equivalence_summary above."
)

print(
    "All N16-N19 cross-notebook equivalence checks passed."
)

Max raw event-weight difference: 6.610575125520768e-06
Max raw ordinary-response-ratio difference: 3.552713678800501e-15


,component,reference_notebook,comparison,n_compared,max_abs_difference,tolerance,passed,notes
0,n,N16,order,1,0.000000e+00,0.000000e+00,True,order
1,m,N16,order,1,0.000000e+00,0.000000e+00,True,order
2,retain_Q_to_R,N16,edge,1,0.000000e+00,0.000000e+00,True,edge
3,retain_I_to_R,N16,edge,1,0.000000e+00,0.000000e+00,True,edge
4,retain_Q_to_I,N16,edge,1,0.000000e+00,0.000000e+00,True,edge
5,retain_R_to_I,N16,edge,1,0.000000e+00,0.000000e+00,True,edge
6,validation_R,N17,ordinary forecasts,1181,9.974660e-17,1.000000e-10,True,ordinary forecasts
7,validation_I,N17,ordinary forecasts,1087,1.686151e-15,1.000000e-10,True,ordinary forecasts
8,test_R,N17,ordinary forecasts,1182,9.974660e-17,1.000000e-10,True,ordinary forecasts
9,test_I,N17,ordinary forecasts,1071,2.400857e-15,1.000000e-10,True,ordinary forecasts


All N16-N19 cross-notebook equivalence checks passed.


## 11. Reuse-mode demonstration

In [13]:
# Deterministic reuse smoke test: alphabetical first eligible audited USD/JPY event outside canonical labels with >=20 rows.
raw=pd.read_csv(ROOT/"Data"/"interim"/"g10_usd_jpy_events_audited.csv");raw["event_datetime_utc"]=pd.to_datetime(raw.event_datetime_utc,utc=True,errors="coerce",format="mixed");candidates=[(n,g.copy())for n,g in raw.loc[raw.ccy.isin(["USD","JPY"])&raw.event_datetime_utc.notna()].groupby("event",sort=True)if n not in set(reference_event_history.event_label)and len(g)>=20];reuse_label,reuse_raw=candidates[0];p=prepare_model_panel(model_panel);sessions=p[["model_day"]].copy();sessions["open"]=(sessions.model_day-pd.Timedelta(days=1)+pd.Timedelta(hours=17)).dt.tz_localize("America/New_York");sessions["close"]=(sessions.model_day+pd.Timedelta(hours=17)).dt.tz_localize("America/New_York")
def session_day(ts):
 z=sessions.loc[(sessions.open<=ts)&(ts<sessions.close),"model_day"];return z.iloc[0]if len(z)else pd.NaT
reuse_raw["realised_model_day"]=reuse_raw.event_datetime_utc.map(session_day);reuse_raw=reuse_raw.loc[reuse_raw.realised_model_day.notna()].copy();prev=pd.Series(p.model_day.iloc[:-1].to_numpy(),index=p.model_day.iloc[1:]);ny=reuse_raw.event_datetime_utc.dt.tz_convert("America/New_York");cut=(reuse_raw.realised_model_day.dt.normalize()+pd.Timedelta(hours=10)).dt.tz_localize("America/New_York");reuse_raw["implied_model_day"]=reuse_raw.realised_model_day;reuse_raw.loc[ny.lt(cut),"implied_model_day"]=reuse_raw.loc[ny.lt(cut),"realised_model_day"].map(prev);reuse_raw=reuse_raw.loc[reuse_raw.implied_model_day.notna()]
reuse_events=pd.DataFrame({"event_id":[f"REUSE_{i:04d}"for i in range(len(reuse_raw))],"event_label":reuse_label,"event_timestamp":reuse_raw.event_datetime_utc,"realised_model_day":reuse_raw.realised_model_day,"implied_model_day":reuse_raw.implied_model_day});reuse_results=estimate_event_weights(model_panel,reuse_events,structure=reference_results["structure"],flexible_spec=reference_results["flexible_spec"],config=CONFIG);assert reuse_results["run_metadata"].loc[0,"structure_source"]=="SUPPLIED"and reuse_results["run_metadata"].loc[0,"flexible_spec_source"]=="SUPPLIED"and len(reuse_results["pooled_weights"]);reuse_audit=pd.DataFrame([{"reuse_event_family":reuse_label,"n_economic_occurrences":len(reuse_events),"passed":True}])


## 12. Compact exports and final audit

In [14]:
api_specification=pd.DataFrame([("event_alignment_version","TARGET_SPECIFIC_NY_CUT_1000_ET"),("event_alignment_source","NOTEBOOK_18"),("alignment_audit","18A_iv_event_alignment_audit.ipynb"),("alignment_selection_sample","TRAIN_ONLY"),("realised_event_day_field","realised_model_day"),("implied_event_day_field","implied_model_day"),("target_day_field","target_model_day"),("bloomberg_snapshot_time_identified","False"),("iv_specification","WEEKDAY_NORM_IV"),("iv_column","iv_model_var"),("n_m","n=1; m=12"),("eventless_history","lags before target-specific masks"),("denominator","strict positive; no repair"),("ordinary_ratios","VALIDATION future discovery calibration; TEST OOS diagnostics only"),("pooling","stage-safe"),("reuse","all models and scalers refit")],columns=["item","value"]).assign(section="API")
exports_20={"20_api_equivalence_summary.csv":equivalence_summary,"20_api_reference_event_weights.csv":reference_results["event_weights"],"20_api_event_mapping.csv":reference_results["event_mapping"],"20_api_ordinary_response_ratios.csv":reference_results["ordinary_response_ratios"],"20_api_ordinary_response_ratio_summary.csv":reference_results["ordinary_response_ratio_summary"],"20_api_run_metadata.csv":reference_results["run_metadata"],"20_api_reuse_audit.csv":reuse_audit,"20_api_specification.csv":api_specification}
for n,t in exports_20.items():t.to_csv(PROCESSED/n,index=False)


In [15]:
assert reference_results["parametric_eventless"]["metadata"]["eventless_parameters_refit"]and reference_results["flexible_eventless"]["metadata"]["eventless_parameters_refit"]and reference_results["flexible_eventless"]["metadata"]["scalers_refit"];assert reference_results["run_metadata"].loc[0,"event_alignment_version"]=="TARGET_SPECIFIC_NY_CUT_1000_ET"and reuse_audit.passed.all();assert exercise_5_pipeline_complete
for n,t in exports_20.items():t.to_csv(PROCESSED/n,index=False)


# NOTEBOOK 20 COMPLETE — EXERCISE 5 VALIDATED

Notebook 20 has been executed successfully from top to bottom in a fresh
kernel. The N16–N19 reference-equivalence checks and the final API audit
all passed.

The reusable Exercise 5 pipeline is complete and ready for Exercise 6.